# Апофеоз ЛР_01–06 · Интерактивный стенд по всем заданиям

Шесть лабораторных — 41 задание — каждое с **полями ввода и живым
пересчётом**: двигаете ползунок — формула задания считается заново,
график и числовые проверки обновляются. Внизу — **сводная сцена**,
где всё собрано в один контур «камера → трекинг → рука → IK», и
автотест, который жёстко сверяет стенд с теорией.

| Лаба | Тема |
|---|---|
| 01 | SE(3), однородные координаты, дерево кадров |
| 02 | SO(3): exp/log, кватернионы, slerp, gimbal lock, дрейф |
| 03 | Модель камеры, дисторсия, стереопара, глубина |
| 04 | Гомография, PnP, Гаусс—Ньютон, RANSAC |
| 05 | DH-параметры, PoE, рабочая зона, символьная FK |
| 06 | IK: Ньютон, аналитика 2R, бассейны, граница, ядро |

Требования: `numpy`, `matplotlib`, `ipywidgets` (в docker-окружении курса есть).

In [1]:
# Общая математика шести лабораторных
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

# ---------- повороты и SE(3) ----------
def Rz(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

def Ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])

def Rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])

def so3_exp(w):
    """Формула Родрига (ЛР_02 задание 1)."""
    th = float(np.linalg.norm(w))
    if th < 1e-12:
        return np.eye(3)
    k = w / th
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * (K @ K)

def so3_log(R):
    """Обратное отображение (ЛР_02 задание 1)."""
    c = (np.trace(R) - 1) / 2
    th = np.arccos(np.clip(c, -1, 1))
    if th < 1e-9:
        return np.zeros(3)
    if np.pi - th < 1e-6:                      # околопи: ветвь через диагональ
        i = int(np.argmax(np.diag(R)))
        k = np.zeros(3)
        k[i] = np.sqrt(max(0.0, (R[i, i] + 1) / 2))
        j, l = (i + 1) % 3, (i + 2) % 3
        k[j] = (R[i, j] + R[j, i]) / (4 * k[i]) if k[i] > 1e-9 else 0
        k[l] = (R[i, l] + R[l, i]) / (4 * k[i]) if k[i] > 1e-9 else 0
        return th * k
    return th / (2 * np.sin(th)) * np.array(
        [R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1]])

def euler_zyx(roll, pitch, yaw):
    """Углы Эйлера (ЛР_02 задания 4-5): композиция Z-Y-X."""
    return Rz(yaw) @ Ry(pitch) @ Rx(roll)

def R_to_quat(R):
    """Матрица -> кватернион (w, x, y, z), ветвь по максимуму диагонали (ЛР_02 з.2)."""
    tr = np.trace(R)
    if tr > 0:
        S = np.sqrt(tr + 1.0) * 2
        q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S,
             (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = np.sqrt(1 + R[0, 0] - R[1, 1] - R[2, 2]) * 2
        q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S,
             (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = np.sqrt(1 + R[1, 1] - R[0, 0] - R[2, 2]) * 2
        q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S,
             (R[1, 2] + R[2, 1]) / S]
    else:
        S = np.sqrt(1 + R[2, 2] - R[0, 0] - R[1, 1]) * 2
        q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S,
             (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.asarray(q, float)
    return q / np.linalg.norm(q)

def quat_to_R(q):
    """Кватернион (w, x, y, z) -> матрица (ЛР_02 задание 2)."""
    w, x, y, z = np.asarray(q, float) / np.linalg.norm(q)
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
        [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
        [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)],
    ])

def slerp(q0, q1, t):
    """Сферическая интерполяция (ЛР_02 задание 5), с учётом двойного накрытия."""
    q0 = q0 / np.linalg.norm(q0)
    q1 = q1 / np.linalg.norm(q1)
    d = float(q0 @ q1)
    if d < 0:
        q1, d = -q1, -d
    if d > 0.9995:
        q = (1 - t) * q0 + t * q1
        return q / np.linalg.norm(q)
    th = np.arccos(np.clip(d, -1, 1))
    return (np.sin((1 - t) * th) * q0 + np.sin(t * th) * q1) / np.sin(th)

def draw_frame(ax, T, name, L=0.15):
    o = T[:3, 3]
    R = T[:3, :3]
    for i, col in enumerate(['#d64545', '#3f9142', '#3b6fb5']):
        e = o + R @ (np.eye(3)[i] * L)
        ax.plot(*zip(o, e), color=col, lw=2)
        tip = o + R @ (np.eye(3)[i] * (L + 0.04))
        label = name[i] if i < len(name) else ''
        ax.text(*tip, label, fontsize=7, color=col)

# ---------- камера (ЛР_03) ----------
F_PX, W_IMG, H_IMG, CX, CY = 800.0, 1280, 720, 640.0, 360.0
BASE_STEREO = 0.12
K = np.array([[F_PX, 0, CX], [0, F_PX, CY], [0, 0, 1]])

def look_at(eye, target, up=(0.0, 0.0, 1.0)):
    eye, target, up = map(lambda v: np.asarray(v, float), (eye, target, up))
    z = target - eye; z /= np.linalg.norm(z)
    x = np.cross(up, z); x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.vstack([x, y, z])
    return R, -R @ eye

def project(K, R, t, X):
    X = np.atleast_2d(np.asarray(X, float))
    Xc = X @ R.T + t
    uvw = Xc @ K.T
    return uvw[:, :2] / uvw[:, 2:3], Xc[:, 2]

def distort_px(uv, k1=-0.28, k2=0.09, p1=1e-3, p2=-8e-4):
    xn = (np.atleast_2d(uv) - [CX, CY]) / F_PX
    x, y = xn[:, 0], xn[:, 1]
    r2 = x * x + y * y
    rad = 1 + k1 * r2 + k2 * r2 * r2
    xd = x * rad + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
    yd = y * rad + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
    return np.column_stack([xd, yd]) * F_PX + [CX, CY]

# ---------- гомография, PnP, RANSAC (ЛР_04) ----------
def dlt_homography(src, dst):
    A = []
    for (x, y), (u, v) in zip(src, dst):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    h = np.linalg.svd(np.asarray(A, float))[2][-1]
    H = h.reshape(3, 3)
    return H / H[2, 2]

def apply_H(H, pts):
    p = np.hstack([np.atleast_2d(pts), np.ones((len(np.atleast_2d(pts)), 1))])
    q = p @ H.T
    return q[:, :2] / q[:, 2:3]

def normalize_pts(p):
    c = p.mean(0)
    s = np.sqrt(2) / np.sqrt(((p - c) ** 2).sum(1)).mean()
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    return apply_H(T, p), T

def dlt_normalized(src, dst):
    sn, Ts = normalize_pts(src)
    dn, Td = normalize_pts(dst)
    Hn = dlt_homography(sn, dn)
    H = np.linalg.inv(Td) @ Hn @ Ts
    return H / H[2, 2]

def pose_from_homography(H, K):
    Ki = np.linalg.inv(K)
    h1, h2, h3 = (Ki @ H).T
    lam = 1.0 / np.linalg.norm(h1)
    r1, r2, t = lam * h1, lam * h2, lam * h3
    R = np.column_stack([r1, r2, np.cross(r1, r2)])
    U, _, Vt = np.linalg.svd(R)
    R = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
    if t[2] < 0:
        R, t = -R, -t
        R[:, 2] *= -1
    return R, t

def ransac_homography(src, dst, thr=3.0, iters=2000, seed=0):
    r = np.random.default_rng(seed)
    best_in, best_H = None, None
    n = len(src)
    for _ in range(iters):
        idx = r.choice(n, 4, replace=False)
        with np.errstate(all='ignore'):
            try:
                Hh = dlt_normalized(src[idx], dst[idx])
            except np.linalg.LinAlgError:
                continue
            d = np.linalg.norm(apply_H(Hh, src) - dst, axis=1)
        inl = d < thr
        if best_in is None or inl.sum() > best_in.sum():
            best_in, best_H = inl, Hh
    if best_in.sum() >= 4:
        best_H = dlt_normalized(src[best_in], dst[best_in])
    return best_H, best_in

def pnp_gauss_newton(Xw, obs, x0, iters=25):
    """Уточнение позы (ЛР_04 задание 4): xi = (w, t), невязка - репроекция."""
    xi = np.asarray(x0, float).copy()
    hist = []
    for _ in range(iters):
        R = so3_exp(xi[:3])
        uv, _ = project(K, R, xi[3:], Xw)
        r = (uv - obs).ravel()
        hist.append(np.linalg.norm(r))
        Jn = np.zeros((len(r), 6))
        h = 1e-7
        for j in range(6):
            xp = xi.copy(); xp[j] += h
            up, _ = project(K, so3_exp(xp[:3]), xp[3:], Xw)
            Jn[:, j] = ((up - obs).ravel() - r) / h
        xi = xi - np.linalg.lstsq(Jn, r, rcond=None)[0]
    return xi, np.array(hist)

# ---------- DH, планарные 2R/3R (ЛР_05/06) ----------
def dh(a, alpha, d, theta):
    ca, sa = np.cos(alpha), np.sin(alpha)
    ct, st = np.cos(theta), np.sin(theta)
    return np.array([
        [ct, -st * ca, st * sa, a * ct],
        [st, ct * ca, -ct * sa, a * st],
        [0.0, sa, ca, d],
        [0.0, 0.0, 0.0, 1.0]])

L2_1, L2_2 = 0.6, 0.4                     # планарный 2R из ЛР_06
L3_ARM = np.array([0.5, 0.35, 0.2])       # планарный 3R из ЛР_05

def fk2(q):
    return np.array([L2_1 * np.cos(q[0]) + L2_2 * np.cos(q[0] + q[1]),
                     L2_1 * np.sin(q[0]) + L2_2 * np.sin(q[0] + q[1])])

def jac2(q):
    return np.array([
        [-L2_1 * np.sin(q[0]) - L2_2 * np.sin(q[0] + q[1]),
         -L2_2 * np.sin(q[0] + q[1])],
        [L2_1 * np.cos(q[0]) + L2_2 * np.cos(q[0] + q[1]),
         L2_2 * np.cos(q[0] + q[1])]])

def ik2_analytic(target, l1=L2_1, l2=L2_2):
    """Две ветви планарного 2R (ЛР_06 задание 2); None, если цель недостижима."""
    x, y = target
    d2 = x * x + y * y
    if d2 > (l1 + l2) ** 2 or d2 < (l1 - l2) ** 2:
        return None
    c2 = np.clip((d2 - l1 ** 2 - l2 ** 2) / (2 * l1 * l2), -1, 1)
    out = []
    for sign in (1, -1):
        th2 = sign * np.arccos(c2)
        th1 = np.arctan2(y, x) - np.arctan2(l2 * np.sin(th2),
                                            l1 + l2 * np.cos(th2))
        out.append(np.array([th1, th2]))
    return out

def ik2_newton(target, q0, damp=0.0, iters=60, tol=1e-12):
    """Ньютон (ЛР_06 задание 3), опционально демпфированный (з.6)."""
    q = np.asarray(q0, float).copy()
    hist = []
    for _ in range(iters):
        r = fk2(q) - target
        J = jac2(q)
        hist.append(np.linalg.norm(r))
        if hist[-1] < tol:
            break
        if damp > 0:
            dq = -J.T @ np.linalg.solve(J @ J.T + damp ** 2 * np.eye(2), r)
        else:
            dq = -np.linalg.pinv(J) @ r
        q = q + dq
    hist.append(np.linalg.norm(fk2(q) - target))
    return q, np.array(hist)

def fk3_planar(q, l=L3_ARM):
    a = np.cumsum(q)
    return np.array([np.sum(l * np.cos(a)), np.sum(l * np.sin(a))])

def jac3_planar(q, l=L3_ARM):
    a = np.cumsum(q)
    return np.array([[-np.sum(l[k:] * np.sin(a[k:])) for k in range(3)],
                     [np.sum(l[k:] * np.cos(a[k:])) for k in range(3)]])

# ---------- пространственная рука (сводная сцена, ЛР_05/06) ----------
D_BASE = 0.30
L_ARM = np.array([0.45, 0.35, 0.15])
REACH = float(np.sum(L_ARM))

def arm_fk(q):
    q1, q2, q3, q4 = q
    p_base = np.array([0.0, 0.0, D_BASE])
    Rz1 = Rz(q1)
    Rs = Rz1 @ Ry(q2 - np.pi / 2)
    p_elbow = p_base + Rs @ np.array([L_ARM[0], 0, 0])
    Re = Rs @ Ry(q3)
    p_wrist = p_elbow + Re @ np.array([L_ARM[1], 0, 0])
    Rw = Re @ Ry(q4)
    p_ee = p_wrist + Rw @ np.array([L_ARM[2], 0, 0])
    points = np.array([[0, 0, 0], p_base, p_elbow, p_wrist, p_ee], float)
    frames = []
    for p, R in ((p_base, Rs), (p_elbow, Re), (p_wrist, Rw)):
        T = np.eye(4)
        T[:3, :3], T[:3, 3] = R, p
        frames.append(T)
    axes = [np.array([0.0, 0.0, 1.0]), Rz1 @ np.array([0.0, 1.0, 0.0]),
            Rz1 @ np.array([0.0, 1.0, 0.0]), Rz1 @ np.array([0.0, 1.0, 0.0])]
    origins = [np.array([0.0, 0.0, 0.0]), p_base, p_elbow, p_wrist]
    return points, frames, axes, origins

def arm_jac(q):
    points, _, axes, origins = arm_fk(q)
    p_ee = points[-1]
    return np.column_stack([np.cross(a, p_ee - o) for a, o in zip(axes, origins)])

def ik3d_newton(target, q0, iters=120, tol=1e-12):
    q = np.asarray(q0, float).copy()
    hist, conds = [], []
    for _ in range(iters):
        r = arm_fk(q)[0][-1] - target
        J = arm_jac(q)
        conds.append(np.linalg.cond(J))
        hist.append(np.linalg.norm(r))
        if hist[-1] < tol:
            break
        q = (q - np.linalg.pinv(J) @ r + np.pi) % (2 * np.pi) - np.pi
    hist.append(np.linalg.norm(arm_fk(q)[0][-1] - target))
    return q, np.array(hist), np.array(conds)

# ---------- сцена ----------
MARKER_C = np.array([0.4, 0.4, 0.0])
MARKER_H = 0.10
EYE_A = np.array([1.4, -1.6, 1.3])
EYE_B = np.array([-1.8, -0.8, 1.9])
R_A, T_A = look_at(EYE_A, MARKER_C)
R_B, T_B = look_at(EYE_B, MARKER_C)
Q_A, Q_B = R_to_quat(R_A), R_to_quat(R_B)
R0, T0 = R_A, T_A

def cam_pose(t):
    R = quat_to_R(slerp(Q_A, Q_B, t))
    eye = (1 - t) * EYE_A + t * EYE_B
    return R, -R @ eye, eye

# ---------- каркас интерактивного блока задания ----------
import ipywidgets as W

def task_block(title, note, controls, fn):
    """Каркас: заголовок + строка ползунков + живой вывод (график/печать)."""
    out = W.Output()

    def run(_=None):
        out.clear_output(wait=True)
        with out:
            fn()

    for c in controls:
        c.observe(run, names='value')
    disp = W.VBox([W.HTML('<b>%s</b>&nbsp;&nbsp;<span style="color:#888">%s</span>'
                          % (title, note)),
                   W.HBox(controls), out])
    display(disp)
    run()

def ok(cond, msg):
    print(('  OK  ' if cond else ' FAIL ') + msg)

print('Общая математика загружена: 6 лабораторных, %d функций' %
      len([f for f in dir() if not f.startswith('_')]))

Общая математика загружена: 6 лабораторных, 68 функций


## ЛР_01 · Однородные координаты и группа SE(3)

Задания 1–7: двигайте входы — матрицы, проверки и триэдры пересчитываются.

In [2]:
# ---------- ЛР_01, задания 1-7 (интерактивно) ----------

# Задание 1: элементарные повороты -> однородная матрица
_w = [W.FloatSlider(value=0.4, min=-3.14, max=3.14, step=0.01, description='yaw (z)'),
      W.FloatSlider(value=-0.3, min=-3.14, max=3.14, step=0.01, description='pitch (y)'),
      W.FloatSlider(value=0.2, min=-3.14, max=3.14, step=0.01, description='roll (x)'),
      W.FloatSlider(value=0.5, min=-2, max=2, step=0.1, description='tx'),
      W.FloatSlider(value=-0.2, min=-2, max=2, step=0.1, description='ty'),
      W.FloatSlider(value=0.8, min=-2, max=2, step=0.1, description='tz')]

def _lab1_t1(_w=_w):
    yaw, pitch, roll = (s.value for s in _w[:3])
    t = np.array([s.value for s in _w[3:]])
    R = Rz(yaw) @ Ry(pitch) @ Rx(roll)
    T = np.eye(4); T[:3, :3], T[:3, 3] = R, t
    print('T =\n', T)
    ok(np.abs(R.T @ R - np.eye(3)).max() < 1e-12, 'R^T R = I (поворот)')
    ok(np.linalg.det(R) > 0 and abs(np.linalg.det(R) - 1) < 1e-12, 'det R = +1')
    print()
    print('Вывод: yaw=%.2f, pitch=%.2f, roll=%.2f, перенос (%.1f, %.1f, %.1f): '
          'R^T R совпала с единичной до %.0e и det=+1 — при любых значениях '
          'ползунков композиция остаётся честным элементом SE(3); однородная '
          'форма упаковывает поворот и перенос в одно умножение.'
          % (yaw, pitch, roll, t[0], t[1], t[2],
             np.abs(R.T @ R - np.eye(3)).max()))

task_block('Задание 1. Элементарные повороты и класс Transform',
           'композиция Rz*yaw * Ry*pitch * Rx*roll + перенос — живая матрица и проверки',
           _w, _lab1_t1)

# Задание 2: точка и вектор при преобразовании
_w = [W.FloatSlider(value=0.5, min=-2, max=2, step=0.1, description='yaw'),
      W.FloatSlider(value=0.3, min=-2, max=2, step=0.1, description='px'),
      W.FloatSlider(value=-0.4, min=-2, max=2, step=0.1, description='py'),
      W.FloatSlider(value=0.6, min=-2, max=2, step=0.1, description='pz')]

def _lab1_t2(_w=_w):
    p = np.array([s.value for s in _w[1:]])
    T = np.eye(4); T[:3, :3] = Rz(_w[0].value); T[:3, 3] = 0
    q = T[:3, :3] @ p + T[:3, 3]
    print('точка p =', p, '->  T*p =', q)
    ok(abs(np.linalg.norm(q) - np.linalg.norm(p)) < 1e-12,
       'поворот сохраняет длину (||T*p|| = ||p||)')
    fig = plt.figure(figsize=(4.6, 3.4))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(*zip(np.zeros(3), p), color='0.4', lw=2)
    ax.plot(*zip(np.zeros(3), q), color='#d64545', lw=2)
    ax.scatter(*zip(p), color='0.4'); ax.scatter(*zip(q), color='#d64545')
    ax.set_title('вектор-радиус до/после поворота')
    plt.tight_layout(); plt.show()
    n_p, n_q = np.linalg.norm(p), np.linalg.norm(q)
    print()
    if n_p > 1e-9:
        ang = np.degrees(np.arccos(np.clip(p @ q / (n_p * n_q), -1, 1)))
        print('Вывод: при yaw=%.2f вектор длины %.3f м повернулся на %.1f град, '
              'длина %.3f м не изменилась: поворот меняет направление, но не '
              'растягивает — поэтому ориентации и хранят ортогональными матрицами.'
              % (_w[0].value, n_p, ang, n_q))
    else:
        print('Вывод: нулевой вектор остаётся нулевым — общая точка всех поворотов.')

task_block('Задание 2. Точка и вектор',
           'направление меняется, длина - нет', _w, _lab1_t2)

# Задание 3: свойства группы
_w = [W.FloatSlider(value=0.5, min=-2, max=2, step=0.05, description='a1'),
      W.FloatSlider(value=-0.3, min=-2, max=2, step=0.05, description='a2'),
      W.FloatSlider(value=0.4, min=-2, max=2, step=0.05, description='b1'),
      W.FloatSlider(value=0.9, min=-2, max=2, step=0.05, description='b2')]

def _lab1_t3(_w=_w):
    A = Rz(_w[0].value) @ Ry(_w[1].value)
    B = Rz(_w[2].value) @ Ry(_w[3].value)
    I = np.eye(3)
    ok(np.abs(A @ np.linalg.inv(A) - I).max() < 1e-12, 'A * A^-1 = I (обратный есть)')
    ok(np.abs((A @ B) @ np.linalg.inv(A @ B) - I).max() < 1e-12, '(AB)(AB)^-1 = I (замкнутость)')
    comm = np.abs(A @ B - B @ A).max()
    if comm > 1e-6:
        print('  OK   порядок важен: AB != BA, |AB - BA| = %.3f (группа не абелева)' % comm)
    else:
        print('  ...  при этих углах повороты коммутируют (одна ось) - сдвиньте ползунки')
    ok(abs(np.linalg.det(A @ B) - np.linalg.det(A) * np.linalg.det(B)) < 1e-12,
       'det(AB) = det(A)det(B)')
    print()
    print('Вывод: при a1=%.2f, a2=%.2f, b1=%.2f, b2=%.2f получилось |AB - BA| = '
          '%.3f: %s. Обратная есть, произведение замкнуто, det мультипликативен — '
          'все групповые аксиомы выполняются на ваших числах, а вот коммутативность '
          '— нет.'
          % (_w[0].value, _w[1].value, _w[2].value, _w[3].value, comm,
             'порядок поворотов меняет результат'
             if comm > 1e-6 else 'повороты легли на одну ось (редкий частный случай)'))

task_block('Задание 3. Свойства группы',
           'группа: замкнутость, обратный, единица; группа не абелева',
           _w, _lab1_t3)

# Задание 5: визуализация триэдров (дерево из ЛР_01)
_w = [W.FloatSlider(value=0.6, min=-3, max=3, step=0.05, description='камера yaw'),
      W.FloatSlider(value=-0.4, min=-3, max=3, step=0.05, description='камера pitch'),
      W.FloatSlider(value=0.3, min=-3, max=3, step=0.05, description='объект yaw')]

def _lab1_t5(_w=_w):
    Tw = np.eye(4)
    Tc = np.eye(4); Tc[:3, :3] = Rz(_w[0].value) @ Ry(_w[1].value); Tc[:3, 3] = [1.0, 0.1, 0.4]
    To = Tc @ np.eye(4)
    To[:3, :3] = Tc[:3, :3] @ Rz(_w[2].value); To[:3, 3] = Tc[:3, 3] + Tc[:3, :3] @ [0.5, -0.2, -0.3]
    fig = plt.figure(figsize=(5.4, 4))
    ax = fig.add_subplot(111, projection='3d')
    draw_frame(ax, Tw, 'xyz', L=0.25)
    draw_frame(ax, Tc, 'xyz', L=0.18)
    draw_frame(ax, To, 'xyz', L=0.12)
    ax.set_title('мир -> камера -> объект (словарь PARENT)')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1); ax.set_zlim(-0.5, 0.8)
    plt.tight_layout(); plt.show()
    print()
    print('Вывод: объект-yaw=%.2f: поза объекта = поза камеры, домноженная на его '
          'поворот и сдвиг в системе камеры. Два кадра связаны не «напрямую», а '
          'через дерево (общий родитель — мир): ровно это делает lookupTransform '
          'в tf2, и потому в ROS хранят словарь PARENT, а не все пары матриц.'
          % _w[2].value)

task_block('Задание 5. Визуализация дерева кадров',
           'триэдры мира, камеры и объекта; объект добиться через камеру',
           _w, _lab1_t5)

# Задание 6: накопление погрешности
_w = [W.IntSlider(value=200, min=10, max=3000, step=10, description='шагов N'),
      W.FloatLogSlider(value=1e-8, base=10, min=-12, max=-6, step=0.5,
                       description='шум eps')]

def _lab1_t6(_w=_w):
    N, eps = _w[0].value, _w[1].value
    R = np.eye(3)
    errs = []
    r = np.random.default_rng(0)
    for k in range(N):
        R = R @ (np.eye(3) + r.normal(0, eps, (3, 3)))    # «ошибочная» арифметика
        errs.append(np.abs(R.T @ R - np.eye(3)).max())
    errs = np.array(errs)
    print('отклонение от ортогональности после %d шагов: %.2e (машинная eps ~ %.0e)'
          % (N, errs[-1], 2.2e-16))
    ok(errs[-1] > 1e-13, 'погрешность накопилась: рост систематический, а не случайный')
    slope = np.polyfit(np.arange(1, N + 1), errs, 1)[0]
    print()
    print('Вывод: N=%d шагов при шуме %.0e — отклонение доросло до %.2e, наклон '
          'прямой %.1e/шаг: ошибка копится линейно, а не «шумит» вокруг нуля. '
          'Длинные цепочки кадров без перепроверки обречены — чем лечить, '
          'показывает задание 7.'
          % (N, eps, errs[-1], slope))
    plt.semilogy(np.arange(1, N + 1), errs)
    plt.xlabel('шаг композиции'); plt.ylabel('max |R^T R - I|')
    plt.title('ЛР_01 задание 6: погрешность цепочки растёт линейно по шагам')
    plt.grid(True); plt.tight_layout(); plt.show()

task_block('Задание 6. Накопление погрешности',
           'цепочка из N случайно возмущённых поворотов — лог-график дрейфа',
           _w, _lab1_t6)

# Задание 7: ортогонализация через SVD
_w = [W.FloatSlider(value=0.01, min=0, max=0.2, step=0.002, description='возмущение')]

def _lab1_t7(_w=_w):
    e = _w[0].value
    R = Rz(0.5) @ Ry(0.3)
    Rd = R + e * np.array([[1, -0.5, 0.3], [0.2, 1, -0.6], [-0.8, 0.4, 1]])
    U, _, Vt = np.linalg.svd(Rd)
    Rfix = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
    print('до:  max|R^T R - I| = %.2e,  det = %+.6f'
          % (np.abs(Rd.T @ Rd - np.eye(3)).max(), np.linalg.det(Rd)))
    print('после SVD: max|R^T R - I| = %.2e,  det = %+.6f'
          % (np.abs(Rfix.T @ Rfix - np.eye(3)).max(), np.linalg.det(Rfix)))
    print('угловая цена проекции: %.2e рад' % np.linalg.norm(so3_log(Rfix @ R.T)))
    ok(np.abs(Rfix.T @ Rfix - np.eye(3)).max() < 1e-12, 'SVD вернул поворот')
    print()
    print('Вывод: возмущение %.3f развалило ортогональность до %.1e — того же '
          'порядка, что само e; SVD вернул матрицу в SO(3) с точностью %.0e, '
          'заплатив %.1e рад (это ближайший поворот). Цена проекции мала против '
          'привнесённой ошибки, поэтому ту же процедуру применяют и в PnP '
          '(ЛР_04, задание 3).'
          % (e, np.abs(Rd.T @ Rd - np.eye(3)).max(),
             np.abs(Rfix.T @ Rfix - np.eye(3)).max(),
             np.linalg.norm(so3_log(Rfix @ R.T))))

task_block('Задание 7. Ортогонализация через SVD',
           'испорченная матрица -> ближайший поворот; та же проекция работает в PnP (ЛР_04)',
           _w, _lab1_t7)

print()
print('Замечание: задание 4 (дерево) и требование world_from/lookup целиком живут в сводной сцене внизу.')


Замечание: задание 4 (дерево) и требование world_from/lookup целиком живут в сводной сцене внизу.


## ЛР_02 · SO(3): экспоненциальное отображение, кватернионы, интерполяция

In [3]:
# ---------- ЛР_02, задания 1-7 (интерактивно) ----------

# Задание 1: exp/log
_w = [W.FloatSlider(value=1.2, min=-3, max=3, step=0.05, description='wx'),
      W.FloatSlider(value=-0.5, min=-3, max=3, step=0.05, description='wy'),
      W.FloatSlider(value=0.8, min=-3, max=3, step=0.05, description='wz')]

def _lab2_t1(_w=_w):
    w = np.array([s.value for s in _w])
    th = float(np.linalg.norm(w))
    R = so3_exp(w)
    print('exp(w): R =\n', R, '\n|w| = %.4f рад = %.1f град' % (th, np.degrees(th)))
    wl = so3_log(R)
    # log возвращает главную ветвь: угол в (-pi, pi]
    th_main = np.arctan2(np.sin(th), np.cos(th))
    w_main = w * (th_main / th) if th > 1e-12 else w
    ok(np.abs(wl - w_main).max() < 1e-10, 'log(exp(w)) = w в главной ветви')
    if th > np.pi + 1e-9:
        print('  ...  |w| = %.2f > pi: exp периодичен, log вернул главное значение %.2f рад'
              % (th, th_main))
    w2 = np.array([0.7, 0.0, 0.0]); w3 = np.array([0.0, 0.7, 0.0])
    ok(np.abs(so3_exp(w2 + w3) - so3_exp(w2) @ so3_exp(w3)).max() > 1e-3,
       'exp не гомоморфизм: exp(w2+w3) != exp(w2)exp(w3)')
    d_nc = np.linalg.norm(so3_exp(w2 + w3) - so3_exp(w2) @ so3_exp(w3))
    print()
    print('Вывод: ваш |w| = %.2f рад — Родригг даёт поворот на %.1f град, log '
          'возвращает главное значение %.2f рад%s. И главное: exp(w2+w3) и '
          'exp(w2)*exp(w3) расходятся на %.2f рад — повороты нельзя складывать '
          'как вектора, угловую скорость интегрируют малыми шагами.'
          % (th, np.degrees(th), th_main,
             ' (вышли за pi — ветвь честно свернулась)' if th > np.pi else '',
             d_nc))

task_block('Задание 1. Экспоненциальное и логарифмическое отображения',
           'формула Родрига и её обращение; проверка главной ветви', _w, _lab2_t1)

# Задание 2: переходы между представлениями
_w = [W.FloatSlider(value=0.5, min=-3, max=3, step=0.05, description='yaw'),
      W.FloatSlider(value=0.3, min=-3, max=3, step=0.05, description='pitch'),
      W.FloatSlider(value=-0.7, min=-3, max=3, step=0.05, description='roll')]

def _lab2_t2(_w=_w):
    R = euler_zyx(_w[2].value, _w[1].value, _w[0].value)
    q = R_to_quat(R)
    print('R -> q(w,x,y,z) = ', q, ' |q| = %.6f' % np.linalg.norm(q))
    ok(np.abs(quat_to_R(q) - R).max() < 1e-12, 'q -> R -> q round trip без потерь')
    print('в ROS geometry_msgs/Quaternion (x,y,z,w) =', np.round(q[[1, 2, 3, 0]], 4))
    print('экспонента: R == quat_to_R([cos(|w|/2), w/|w|*sin(|w|/2)]), w =',
          np.round(so3_log(R), 3))
    err_rt = np.abs(quat_to_R(q) - R).max()
    print()
    print('Вывод: углы (%.2f, %.2f, %.2f) -> q=(w,x,y,z)=(%.3f, %.3f, %.3f, %.3f), '
          '|q|=%.6f, round-trip ошибка %.0e: все представления — один и тот же '
          'поворот. В ROS тот же кватернион лежит как (x,y,z,w)=(%s), и норму '
          'никто за вас не проверяет — нормируйте при передаче.'
          % (_w[0].value, _w[1].value, _w[2].value,
             q[0], q[1], q[2], q[3], np.linalg.norm(q), err_rt,
             np.round(q[[1, 2, 3, 0]], 3)))

task_block('Задание 2. Переходы между представлениями',
           'матрица <-> кватернион <-> ось-угол; порядок ROS x,y,z,w',
           _w, _lab2_t2)

# Задание 3: двойное накрытие
_w = [W.FloatSlider(value=0.9, min=0, max=3.14, step=0.02, description='угол, рад')]

def _lab2_t3(_w=_w):
    th = _w[0].value
    q = np.array([np.cos(th / 2), 0, 0, np.sin(th / 2)])
    ok(np.abs(quat_to_R(q) - quat_to_R(-q)).max() < 1e-15,
       'q и -q дают ОДНУ матрицу (двойное накрытие SO(3) сферой S3)')
    print('углы поворота: у q = %.3f рад, у -q = %.3f рад - а вращение одно'
          % (2 * np.arccos(np.clip(q[0], -1, 1)),
             2 * np.arccos(np.clip(-q[0], -1, 1))))
    t = 0.5
    ok(np.linalg.norm(slerp(q, -q, t) - slerp(q, q * 0 + np.array([np.cos(th / 2), 0, 0, np.sin(th / 2)]), t)) < 1e-15,
       'slerp разрешает накопление сам (кратчайшая дуга)')
    print()
    print('Вывод: при повороте на %.2f рад «внутренний угол» q равен %.1f град, '
          'у -q — %.1f град, а вращение одно (двойное накрытие). slerp сам '
          'выбирает короткую дугу по знаку скалярного произведения — поэтому '
          'при непрерывном трекинге ориентация не прыгает между q и -q.'
          % (th, np.degrees(2 * np.arccos(np.clip(q[0], -1, 1))),
             np.degrees(2 * np.arccos(np.clip(-q[0], -1, 1)))))

task_block('Задание 3. Двойное накрытие',
           'q и -q - один поворот; почему солверы нормируют знак', _w, _lab2_t3)

# Задание 4: gimbal lock
_w = [W.FloatSlider(value=89.5, min=-95, max=95, step=0.5, description='pitch, град')]

def _lab2_t4(_w=_w):
    p = np.radians(_w[0].value)

    def gimbal_measure(pitch, roll=0.0, yaw=0.0):
        # определитель матрицы трёх осей параметров (в мире):
        # yaw - всегда z, pitch - Rz(yaw)*y, roll - Rz(yaw)Ry(pitch)*x
        ax_yaw = np.array([0.0, 0.0, 1.0])
        ax_pitch = Rz(yaw) @ np.array([0.0, 1.0, 0.0])
        ax_roll = Rz(yaw) @ Ry(pitch) @ np.array([1.0, 0.0, 0.0])
        return abs(np.linalg.det(np.column_stack([ax_yaw, ax_pitch, ax_roll])))

    sweep = np.radians(np.linspace(-95, 95, 191))
    meas = np.array([gimbal_measure(pp) for pp in sweep])
    m_now = gimbal_measure(p)
    print('pitch = %.1f град: |det(оси yaw,pitch,roll)| = %.6f, '
          '|cos(pitch)| = %.6f' % (_w[0].value, m_now, abs(np.cos(p))))
    ok(abs(m_now - abs(np.cos(p))) < 1e-9,
       'определитель осей = |cos(pitch)|')
    ok(meas.max() > meas.min() * 100,
       'у ±90 град мера падает на два порядка: оси roll и yaw сливаются')
    plt.semilogy(np.degrees(sweep), np.maximum(meas, 1e-16))
    plt.plot([_w[0].value], [m_now], 'ro')
    plt.xlabel('pitch, град'); plt.ylabel('|det(оси yaw,pitch,roll)|')
    plt.title('ЛР_02 задание 4: вырождение якобиана Эйлеров у +-90 град')
    plt.grid(True); plt.tight_layout(); plt.show()
    print('физика gimbal lock: у ±90 град roll и yaw крутят вокруг одной оси,')
    print('и два параметра перестают быть различимыми')
    print()
    print('Вывод: при вашем pitch=%.1f град |det| = %.4f — %s. Тяните ползунок к '
          '±90 и смотрите обвал на графике в реальном времени: Эйлеры годятся '
          'для отображения и ручного ввода, но не для фильтров и TF.'
          % (_w[0].value, m_now,
             'оси roll и yaw почти слились: теряется степень свободы (gimbal lock)'
             if m_now < 0.15 else 'оси пока различимы, параметризация устойчива'))

task_block('Задание 4. Gimbal lock — численное обнаружение',
           'det якобиана Эйлеров против |cos(pitch)|; тяните pitch к 90 град',
           _w, _lab2_t4)

# Задание 5: SLERP против Эйлеров (разворот не вокруг одной оси Эйлеров -
# иначе линейная интерполяция Эйлеров тоже шла бы по геодезической)
_w = [W.FloatSlider(value=1.7, min=0.3, max=3, step=0.05, description='разворот, рад')]

def _lab2_t5(_w=_w):
    th = _w[0].value
    R1 = euler_zyx(0.35, -0.45, th)          # композиция поворотов вокруг разных осей
    q0 = np.array([1.0, 0, 0, 0])
    q1 = R_to_quat(R1)
    w1 = so3_log(R1)                          # геодезическая: ось фиксирована, угол линейный
    e1 = np.array([0.35, -0.45, th])         # эйлеровы координаты той же позы
    ts = np.linspace(0, 1, 51)
    dev_q, dev_e = [], []
    for t in ts:
        rq = quat_to_R(slerp(q0, q1, t))
        dev_q.append(np.degrees(np.linalg.norm(so3_log(rq) - t * w1)))
        re = euler_zyx(*t * e1)               # «наивная» интерполяция углов Эйлера
        dev_e.append(np.degrees(np.linalg.norm(so3_log(re) - t * w1)))
    print('максимальное отклонение от геодезической:')
    print('  slerp:      %.4f град (ось фиксирована, угол растёт линейно)' % max(dev_q))
    print('  Эйлеры lerp: %.2f град (ось промежуточного поворота «плавает»)' % max(dev_e))
    ok(max(dev_q) < 1e-6, 'slerp идёт по геодезической (кратчайшая дуга)')
    ok(max(dev_e) > 0.1, 'линейная интерполяция Эйлеров с геодезической сбивается')
    plt.plot(ts, dev_q, label='slerp', lw=2)
    plt.plot(ts, dev_e, label='lerp углов Эйлера')
    plt.xlabel('t'); plt.ylabel('отклонение, град'); plt.legend(); plt.grid(True)
    plt.title('ЛР_02 задание 5: интерполяции на одном развороте')
    plt.tight_layout(); plt.show()
    print()
    print('Вывод: на развороте %.2f рад slerp уклонился от геодезической на %.1e '
          'град (ось фиксирована — численный ноль), лерп Эйлеров — на %.2f град, '
          'то есть в %.0f раз сильнее. Обе кривые приходят в ту же позу, но путь '
          'у Эйлеров идёт «не туда»: ось вращения плавает. Для траекторий камер '
          'это и есть разница между плавным и дёрганым облётом.'
          % (th, max(dev_q), max(dev_e), max(dev_e) / max(max(dev_q), 1e-9)))

task_block('Задание 5. SLERP против интерполяции Эйлеров',
           'один разворот ось-z: обе кривые против геодезической', _w, _lab2_t5)

# Задание 6: дрейф при интегрировании (наивный шаг Эйлера vs нормировка)
_w = [W.IntSlider(value=2000, min=100, max=20000, step=100, description='шагов'),
      W.FloatSlider(value=0.5, min=0.05, max=0.9, step=0.01, description='шаг, рад')]

def _lab2_t6(_w=_w):
    N, dt = _w[0].value, _w[1].value
    axis = np.array([0.4, 0.3, 0.5])
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]]) * dt
    R_raw = np.eye(3)          # наивный путь: шаг Эйлера без проекции на SO(3)
    R_nq = np.eye(3)           # тот же шаг + нормировка кватернионом
    ortho_raw, ortho_nq = [], []
    for _ in range(N):
        R_raw = R_raw + K @ R_raw
        R_raw /= np.linalg.norm(R_raw) / np.sqrt(3)   # масштаб держим, ортогональность - нет
        R_nq = quat_to_R(R_to_quat(R_nq + K @ R_nq))
        ortho_raw.append(np.abs(R_raw.T @ R_raw - np.eye(3)).max())
        ortho_nq.append(np.abs(R_nq.T @ R_nq - np.eye(3)).max())
    print('после %d шагов по %.2f рад (всего %.1f оборота):'
          % (N, dt, N * dt / (2 * np.pi)))
    print('  ортогональность наивного пути:   max|R^T R - I| = %.2e' % ortho_raw[-1])
    print('  ортогональность нормированного:  max|R^T R - I| = %.2e' % ortho_nq[-1])
    ok(ortho_nq[-1] < 1e-12,
       'нормированный путь остался честным поворотом на каждом шаге')
    ok(ortho_raw[-1] > 1e-6,
       'наивная матричная арифметика теряет ортогональность (долгий полёт накапливает ошибку)')
    plt.semilogy(np.arange(1, N + 1), ortho_raw, label='наивный путь')
    plt.semilogy(np.arange(1, N + 1), np.maximum(ortho_nq, 1e-17), label='с кватернионной нормировкой')
    plt.xlabel('шаг'); plt.ylabel('max|R^T R - I|'); plt.legend()
    plt.title('ЛР_02 задание 6: дрейф против нормированного пути')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: %d шагов по %.2f рад — суммарно %.0f град (около %.1f оборота): '
          'наивный путь накопил |R^TR - I| = %.1e и дальше только хуже, '
          'нормированный кватернионом держит %.0e на каждом шаге. Без проекции '
          'на SO(3) матричная арифметика непригодна для длительного интегрирования '
          'ориентации (IMU, одометрия).'
          % (N, dt, N * dt * 180 / np.pi, N * dt / (2 * np.pi),
             ortho_raw[-1], max(ortho_nq[-1], 2.2e-16)))

task_block('Задание 6. Дрейф при длительном интегрировании',
           'одна и та же стрелка w*dt: матрицы vs кватернионы с нормировкой',
           _w, _lab2_t6)

# Задание 7: цена восстановления
_w = [W.IntSlider(value=300, min=50, max=1500, step=50, description='матриц N')];

def _lab2_t7(_w=_w):
    N = _w[0].value
    r = np.random.default_rng(0)
    Ms = [euler_zyx(0.1, 0.2, 0.3) + r.normal(0, 0.02, (3, 3)) for _ in range(N)]
    import time
    t0 = time.perf_counter()
    err_svd = 0.0
    for M in Ms:
        U, _, Vt = np.linalg.svd(M)
        Rf = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
        err_svd += np.abs(Rf.T @ Rf - np.eye(3)).max()
    t_svd = (time.perf_counter() - t0) / N * 1e6
    t0 = time.perf_counter()
    err_nrm = 0.0
    for M in Ms:
        q = R_to_quat(M)
        err_nrm += np.abs(quat_to_R(q).T @ quat_to_R(q) - np.eye(3)).max()
    t_nrm = (time.perf_counter() - t0) / N * 1e6
    print('SVD:        %.1f мкс/матрицу, ортогональность %.1e' % (t_svd, err_svd / N))
    print('нормировка: %.1f мкс/матрицу, ортогональность %.1e' % (t_nrm, err_nrm / N))
    ok(t_nrm < t_svd * 3, 'кватернионная нормировка дешевле SVD - выбор ROS-стека')
    print('обе возвращают честные повороты: в ROS ориентацию держат кватернионом')
    print()
    if t_nrm < t_svd:
        tv = 'кватернионная нормировка дешевле в %.1f раза' % (t_svd / t_nrm)
    else:
        tv = 'в этом прогоне времена сопоставимы (шум измерения)'
    print('Вывод: N=%d матриц: %.1f мкс (нормировка) против %.1f мкс (SVD) — %s; '
          'ортогональность после обеих машинная (%.0e и %.0e). Для потока '
          'ориентаций в реальном времени выбирают кватернионы; SVD-проекцию '
          'оставляют редкой санитарной проверкой.'
          % (N, t_nrm, t_svd, tv, err_nrm / N, err_svd / N))

task_block('Задание 7. Цена восстановления: SVD против нормировки',
           'два способа ре-ортогонализации: время и точность', _w, _lab2_t7)

## ЛР_03 · Модель камеры, проективная геометрия и стереопара

In [4]:
# ---------- ЛР_03, задания 1-7 (интерактивно) ----------

# Задание 1: модель камеры
_w = [W.FloatSlider(value=800, min=200, max=2000, step=20, description='f, пикс'),
      W.FloatSlider(value=2.0, min=0.5, max=6, step=0.1, description='Z, м')]

def _lab3_t1(_w=_w):
    f, Z = _w[0].value, _w[1].value
    Ki = np.array([[f, 0, CX], [0, f, CY], [0, 0, 1]])
    # сцена сдвинута на Z вперёд по оптической оси (камера смотрит в +z)
    uv, zc = project(Ki, np.eye(3), [0, 0, Z], [0, 0, 0])
    print('K =\n', Ki)
    print('точка на оптической оси, глубина Zc = %.1f м: uv = %s '
          '- ровно главная точка (%.0f, %.0f)' % (zc[0], uv[0], CX, CY))
    ok(abs(uv[0, 0] - CX) < 1e-9 and abs(uv[0, 1] - CY) < 1e-9,
       'ось -> главная точка')
    uv, _ = project(Ki, np.eye(3), [0, 0, Z], [1.0, 0, 0])
    px = f * (1.0 / Z)
    ok(abs(uv[0, 0] - (CX + f / Z)) < 1e-9,
       'сдвиг на 1 м при Z = %.1f -> %.0f пикс (f/Z); глубина кодируется масштабом'
       % (Z, px))
    print()
    print('Вывод: f=%.0f px, Z=%.1f м -> масштаб %.1f px/м: метр поперёк сцены '
          'занимает %.0f пикс. Тяните Z и смотрите, как px/м падает — дальнее '
          'мельче, и по одному кадру глубину не восстановить: нужен второй взгляд '
          '(стерео, задание 4) или известный размер объекта (PnP, ЛР_04).'
          % (f, Z, px, px))

task_block('Задание 1. Модель камеры и матрица проекции',
           'K из f и главной точки; свойства модели-обскуры', _w, _lab3_t1)

# Задание 2: каркасный рендеринг (куб)
_w = [W.FloatSlider(value=0.0, min=-3, max=3, step=0.05, description='yaw'),
      W.FloatSlider(value=0.5, min=-2, max=5, step=0.1, description='Z, м')]

def _lab3_t2(_w=_w):
    Z = _w[1].value
    s = 0.3
    V = np.array(list(np.ndindex(2, 2, 2))).astype(float) * 2 - 1
    V = V * s
    E = [(i, i + 4) for i in range(4)] + [(0, 1), (1, 3), (3, 2), (2, 0),
                                          (4, 5), (5, 7), (7, 6), (6, 4)]
    Rc, tc = look_at([0.0, -1.0, 0.5], [0.0, 0.0, 0.0])
    Rw = Rz(_w[0].value)
    Xw = V @ Rw.T + [0, 0, Z]
    uv, z = project(K, Rc, tc, Xw)
    plt.figure(figsize=(4.8, 3.2))
    for a, b in E:
        if z[a] > 0 and z[b] > 0:
            plt.plot(*uv[[a, b]].T, color='#1F2733', lw=1.8)
    plt.plot(CX, CY, '+', color='#E08A2E', ms=10)
    plt.title('каркас куба без графических библиотек (ЛР_03 задание 2)')
    plt.gca().set_aspect('equal'); plt.xlim(0, 1280); plt.ylim(720, 0)
    plt.tight_layout(); plt.show()
    px_front = 2 * s * F_PX / max(Z, 0.1)
    print()
    print('Вывод: куб с ребром %.1f м при Z=%.1f м занимает ~%.0f пикс по фронту '
          '(f*s/Z). Каркас собран линейной проекцией без единой 3D-библиотеки: '
          'матрица K развернула вершины в пиксели, а yaw крутит модель в системе '
          'камеры. Отодвигаете куб — он сжимается ровно как 1/Z.'
          % (2 * s, Z, px_front))

task_block('Задание 2. Каркасный рендеринг без графических библиотек',
           'куб проецируется в кадр программно; ось v вниз', _w, _lab3_t2)

# Задание 3: дисторсия
_w = [W.FloatSlider(value=-0.28, min=-0.6, max=0.05, step=0.01, description='k1'),
      W.FloatSlider(value=0.05, min=-0.1, max=0.3, step=0.01, description='k2')]

def _lab3_t3(_w=_w):
    k1, k2 = _w[0].value, _w[1].value
    gg = np.linspace(-0.4, 0.4, 13)
    uv = np.array([[CX + u * F_PX, CY + v * F_PX] for v in gg for u in gg])
    dist = distort_px(uv, k1=k1, k2=k2, p1=0.0, p2=0.0)
    plt.figure(figsize=(5.2, 4))
    for i in range(0, len(uv), 13):
        plt.plot(*uv[i:i + 13].T, color='0.75', lw=0.8)
        plt.plot(*dist[i:i + 13].T, color='#d64545', lw=1.2)
    for j in range(13):
        plt.plot(*uv[j::13].T, color='0.75', lw=0.8)
        plt.plot(*dist[j::13].T, color='#d64545', lw=1.2)
    plt.title('сетка: серая - идеал, красная - дисторсия (k1=%.2f, k2=%.2f)' % (k1, k2))
    plt.gca().set_aspect('equal'); plt.grid(True)
    plt.tight_layout(); plt.show()
    # «вычесть» k1 из уже искажённых пикселей нельзя точно — модель работает в одну сторону
    back = distort_px(dist, -k1 * 0.99, -k2 * 0.99, 0, 0)
    print('у края кадра смещение %.0f пикс; грубое обращение даёт ошибку %.0f пикс'
          % (np.linalg.norm(dist[-1] - uv[-1]),
             np.linalg.norm(back[-1] - dist[-1])))
    disp = np.linalg.norm(dist - uv, axis=1)
    print()
    print('Вывод: k1=%.2f, k2=%.2f — максимальное смещение %.0f пикс на краю '
          'кадра, у центра почти ноль: радиальная дисторсия растёт с радиусом '
          '(члены ~ r^3 и r^5). Знак k1 задаёт «подушку» или «бочку», а обратное '
          'преобразование строят по карте или итерациями — в лоб модель не '
          'обращается, что и показывает строка про ошибку выше.'
          % (k1, k2, disp.max()))

task_block('Задание 3. Дисторсия и её компенсация',
           'k1/k2 гнут сетку; обратное преобразование - компенсация',
           _w, _lab3_t3)

# Задание 4: стереопара и глубина
_w = [W.FloatSlider(value=1.5, min=0.3, max=5, step=0.1, description='Z, м'),
      W.FloatSlider(value=0.12, min=0.02, max=0.5, step=0.01, description='база B, м')]

def _lab3_t4(_w=_w):
    Z, B = _w[0].value, _w[1].value
    X = np.array([0.3, 0.2, 0.0])
    # левая камера в начале координат, сцена на глубине Z по оптической оси
    uvL, z = project(K, np.eye(3), [0, 0, Z], X)
    d = F_PX * B / Z
    uvR = uvL - [d, 0]
    print('левая:  uv = %s (глубина %.2f м)' % (uvL[0].round(2), z[0]))
    print('правая: uv = %s (сдвинута на диспаритет %.1f пикс)' % (uvR[0].round(2), d))
    Zback = F_PX * B / d
    ok(abs(Zback - Z) < 1e-12, 'Z = f*B/d = %.3f м - восстановление точное' % Zback)
    print('при B=%.2f м и Z=%.1f м диспаритет всего %.1f пикс - чем дальше, '
          'тем хуже (задание 5)' % (B, Z, d))
    dz = Z ** 2 / (F_PX * B)
    print()
    print('Вывод: B=%.2f м, Z=%.1f м -> d=%.1f пикс, Z=fB/d сошлось точно. Но цена '
          'пикселя шума: dZ = Z^2/(fB) = %.0f мм на пикс — тяните Z вверх и '
          'увидите квадратичный рост, тяните B — линейное падение. Это же число '
          'определяет СКО глубины в задании 5.'
          % (B, Z, d, dz * 1000))

task_block('Задание 4. Стереопара и восстановление глубины',
           'две проекции -> диспаритет -> Z = fB/d', _w, _lab3_t4)

# Задание 5: погрешность глубины растёт как z^2
_w = [W.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='шум, пикс')]

def _lab3_t5(_w=_w):
    s = _w[0].value
    z = np.linspace(0.5, 6, 30)
    rms_num, rms_th = [], []
    for Z in z:
        d = F_PX * BASE_STEREO / Z
        n = np.random.default_rng(int(Z * 10)).normal(0, s, 4000)
        zz = F_PX * BASE_STEREO / np.maximum(d + n, 1e-9)
        rms_num.append(np.sqrt(np.mean((zz - Z) ** 2)))
        rms_th.append(Z ** 2 * s / (F_PX * BASE_STEREO))
    slope = np.polyfit(np.log(z), np.log(rms_num), 1)[0]
    print('наклон экспериментальной кривой: %.2f (теория: квадрат Z -> 2.00)' % slope)
    ok(abs(slope - 2) < 0.15, 'СКО глубины растёт как Z^2')
    plt.loglog(z, rms_num, 'o-', label='эксперимент')
    plt.loglog(z, rms_th, '--', label='теория z²·s/(fB)')
    plt.xlabel('Z, м'); plt.ylabel('СКО глубины, м'); plt.legend(); plt.grid(True)
    plt.title('ЛР_03 задание 5: погрешность глубины против дистанции')
    plt.tight_layout(); plt.show()
    print()
    print('Вывод: шум %.2f пикс — экспериментальный наклон %.2f (теория: 2). СКО '
          'глубины выросло с %.4f м (Z=%.1f м) до %.4f м (Z=%.1f м) — в %.0f раз '
          'при росте дистанции всего в %.0f раз: закон Z^2 подтверждён вашими '
          'настройками. Стерео слепнет вдаль, и калибровка этого не отменит — '
          'лечится только базой (задание 6).'
          % (s, slope, rms_num[0], z[0], rms_num[-1], z[-1],
             rms_num[-1] / max(rms_num[0], 1e-12), z[-1] / z[0]))

task_block('Задание 5. Погрешность глубины растёт как квадрат расстояния',
           'Монте-Карло при разных Z против аналитики', _w, _lab3_t5)

# Задание 6: влияние базы
_w = [W.FloatSlider(value=2.0, min=0.5, max=6, step=0.1, description='Z, м')]

def _lab3_t6(_w=_w):
    Z = _w[0].value
    Bs = np.logspace(np.log10(0.02), np.log10(0.6), 25)
    rms = [Z ** 2 * 0.25 / (F_PX * b) for b in Bs]
    slope = np.polyfit(np.log(Bs), np.log(rms), 1)[0]
    print('наклон зависимости от базы: %.2f (теория: 1/B -> -1.00)' % slope)
    ok(abs(slope + 1) < 0.05, 'ошибка убывает как 1/B')
    print('чтобы СКО упало вдвое, базу нужно удвоить. Практический предел: '
          'синхронизация кадров и падение общего поля зрения с ростом B')
    plt.loglog(Bs, rms)
    plt.axvline(BASE_STEREO, color='0.6', ls='--')
    plt.text(BASE_STEREO, max(rms), ' база стенда')
    plt.xlabel('база B, м'); plt.ylabel('СКО глубины, м')
    plt.title('ЛР_03 задание 6: больше база - меньше ошибка (до предела)')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: наклон %.2f около -1: СКО ~ 1/B, удвоение базы — вдвое меньше '
          'ошибка. При Z=%.2f м база стенда %.2f м даёт СКО около %.3f м; дальше '
          'растут уже не метры точности, а проблемы: рассинхрон кадров и сузившееся '
          'общее поле зрения — практический предел.'
          % (slope, Z, BASE_STEREO, Z ** 2 * 0.25 / (F_PX * BASE_STEREO)))

task_block('Задание 6. Влияние базы',
           'СКО глубины против B: наклон -1 и практический предел', _w, _lab3_t6)

# Задание 7: триангуляция в общем случае
_w = [W.FloatSlider(value=0.3, min=0, max=1.5, step=0.01, description='шум, пикс')]

def _lab3_t7(_w=_w):
    s = _w[0].value
    X0 = np.array([0.3, 0.2, 1.2])
    Ra, ta = look_at([-0.4, -1.0, 0.4], X0)      # камеры общего положения
    Rb, tb = look_at([0.6, -0.9, 0.8], X0)

    def triangulate(uva, uvb):
        # уравнения камеры: f*xc = (u-cx)*zc -> строка [f*R0 - uc*R2]
        rows, rhs = [], []
        for R, t, uv in ((Ra, ta, uva), (Rb, tb, uvb)):
            uc, vc = uv[0] - CX, uv[1] - CY
            rows += [F_PX * R[0] - uc * R[2], F_PX * R[1] - vc * R[2]]
            rhs += [uc * t[2] - F_PX * t[0], vc * t[2] - F_PX * t[1]]
        return np.linalg.lstsq(np.array(rows), np.array(rhs), rcond=None)[0]

    errs = []
    r = np.random.default_rng(0)
    for _ in range(300):
        uva, _ = project(K, Ra, ta, X0[None])
        uvb, _ = project(K, Rb, tb, X0[None])
        Xh = triangulate(uva[0] + r.normal(0, s, 2), uvb[0] + r.normal(0, s, 2))
        errs.append(np.linalg.norm(Xh - X0))
    errs = np.array(errs)
    Xh0 = triangulate(project(K, Ra, ta, X0[None])[0][0],
                      project(K, Rb, tb, X0[None])[0][0])
    print('без шума: восстановление сходится с ошибкой %.2e м'
          % np.linalg.norm(Xh0 - X0))
    print('300 прогонов при шуме %.2f пикс: медиана %.4f м, СКО %.4f м'
          % (s, np.median(errs), errs.std()))
    print('  (та же схема DLT + решение переопределённой системы, что в ЛР_04)')
    if s < 1e-9:
        ok(np.linalg.norm(Xh0 - X0) < 1e-9, 'нулевая невязка: геометрия точная')
    else:
        ok(np.median(errs) < 0.06,
           'триангуляция сошлась к окрестности точки (ошибка ~ шум*Z/f масштаб)')
    print()
    print('Вывод: без шума восстановление точное (%.0e м); при вашем шуме %.2f '
          'пикс медиана %.4f м — ошибка ~ шум·Z/f, как у стерео. Два произвольных '
          'вида избыточны (4 уравнения на 3 неизвестных), МНК усредняет шум; та же '
          'линеаризация работает и в PnP (ЛР_04, задание 4).'
          % (np.linalg.norm(Xh0 - X0), s, np.median(errs)))

task_block('Задание 7. Триангуляция в общем случае (дополнительное)',
           'две камеры произвольного положения -> DLT-триангуляция точки',
           _w, _lab3_t7)

## ЛР_04 · Гомография, задача PnP и метод RANSAC

In [5]:
# ---------- ЛР_04, задания 1-6 (интерактивно) ----------
g = np.linspace(-0.09, 0.09, 9)
M2 = np.array([[a, b] for a in g for b in g])
Xw = np.column_stack([M2 + MARKER_C[:2], np.zeros(len(M2))])
obs_clean, _ = project(K, R0, T0, Xw)

# Задание 1: DLT-гомография
_w = [W.FloatSlider(value=0.3, min=0, max=2, step=0.05, description='шум, пикс')]

def _lab4_t1(_w=_w):
    s = _w[0].value
    r = np.random.default_rng(0)
    obs = obs_clean + r.normal(0, s, obs_clean.shape)
    H = dlt_homography(M2, obs)
    res = np.linalg.norm(apply_H(H, M2) - obs, axis=1)
    print('H =\n', H)
    print('репроекция: медиана %.3f пикс, max %.3f пикс' % (np.median(res), res.max()))
    if np.median(res) < max(1.0, 1.2 * s):
        ok(True, 'DLT прошёл через все точки (собственный, не cv2)')
    else:
        print('  ...  при шуме %.1f пикс сырой DLT уже разваливается '
              '(медиана %.1f пикс):' % (s, np.median(res)))
        print('        столбцы матрицы плана различаются на порядки - '
              'см. задание 2 (нормировка Хартли)')
    med = float(np.median(res))
    print()
    if med < max(1.0, 1.2 * s):
        print('Вывод: ваш шум %.1f пикс, медиана остатков %.2f пикс — на уровне '
              'самого шума: 162 уравнения (81 точка по 2 координаты) избыточны для '
              '9 неизвестных H, и DLT честно усредняет шум. Поднимите шум до ~2 '
              'пикс и увидите развал — почему, объяснит задание 2.'
              % (s, med))
    else:
        print('Вывод: ваш шум %.1f пикс, медиана остатков %.1f пикс — уже сильно '
              'выше шума: матрица плана плохо обусловлена, решение «плывёт». '
              'Лечится нормировкой Хартли — задание 2.'
              % (s, med))

task_block('Задание 1. Гомография методом DLT',
           'своё DLT + SVD; шум наблюдений регулируется', _w, _lab4_t1)

# Задание 2: нормализация Хартли
_w = [W.FloatSlider(value=0.3, min=0, max=2, step=0.05, description='шум, пикс')]

def _lab4_t2(_w=_w):
    s = _w[0].value
    r = np.random.default_rng(0)
    obs = obs_clean + r.normal(0, s, obs_clean.shape)
    H1 = dlt_homography(M2, obs)
    H2 = dlt_normalized(M2, obs)

    def design_cond(src, dst, normalized):
        # cond настоящей матрицы плана DLT (18x9)
        if normalized:
            src, _ = normalize_pts(src)
            dst, _ = normalize_pts(dst)
        A = []
        for (x, y), (u, v) in zip(src, dst):
            A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
            A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
        return np.linalg.cond(np.array(A, float))

    c1 = design_cond(M2, obs, False)
    c2 = design_cond(M2, obs, True)
    res1 = np.linalg.norm(apply_H(H1, M2) - obs, axis=1)
    res2 = np.linalg.norm(apply_H(H2, M2) - obs, axis=1)
    print('cond матрицы плана DLT: сырая %.1e -> нормированная %.0f '
          '(на %d порядков меньше)' % (c1, c2, round(np.log10(c1 / c2))))
    print('медиана невязки: сырая %.3f пикс, нормированная %.3f пикс'
          % (np.median(res1), np.median(res2)))
    ok(c2 * 1000 < c1, 'нормировка Хартли улучшает обусловленность матрицы плана')
    if s < 0.8:
        ok(np.median(res2) < max(1.0, 1.2 * s),
           'при слабом шуме обе версии точны: выигрыш пока не виден')
    else:
        ok(np.median(res2) < 1.5 * s + 0.5,
           'при шуме %.1f пикс нормированная держит точность, сырая разваливается '
           '(медиана %.1f)' % (s, np.median(res1)))
    print()
    print('Вывод: cond матрицы плана %.1e -> %.0f (на %d порядка меньше) при вашем '
          'шуме %.1f пикс: невязка сырой %.2f пикс против нормированной %.2f. При '
          'слабом шуме точности совпадают (модель та же), при сильном выживает '
          'только нормированная — выигрыш Хартли не в формуле, а в устойчивости.'
          % (c1, c2, round(np.log10(c1 / c2)), s,
             np.median(res1), np.median(res2)))

task_block('Задание 2. Нормализация Хартли',
           'разница cond на порядки; почему точность всё равно похожа',
           _w, _lab4_t2)

# Задание 3: поза из гомографии + куб поверх маркера
_w = [W.FloatSlider(value=0.3, min=-1.5, max=1.5, step=0.05, description='наклон куба')]

def _lab4_t3(_w=_w):
    H = dlt_homography(M2, obs_clean)
    R_est, t_est = pose_from_homography(H, K)
    ok(np.abs(R_est - R0).max() < 1e-9, 'поза из гомографии = точная поза камеры')
    # куб, стоящий на маркере: вершины в мире -> кадр
    s = 0.08
    Vs = np.array(list(np.ndindex(2, 2, 2))).astype(float) * s
    th = _w[0].value
    tilt = Rz(0.0) @ Ry(th)     # наклон куба вокруг метки (для наглядности)
    Vw = Vs @ tilt.T + MARKER_C + np.array([0, 0, 0.001])
    uv, z = project(K, R0, T0, Vw)
    E = [(i, i + 4) for i in range(4)] + [(0, 1), (1, 3), (3, 2), (2, 0),
                                          (4, 5), (5, 7), (7, 6), (6, 4)]
    MS = np.array([[-1, -1, 0], [1, -1, 0], [1, 1, 0], [-1, 1, 0],
                   [-1, -1, 0]]) * MARKER_H + MARKER_C
    uvM, _ = project(K, R0, T0, MS)
    plt.figure(figsize=(4.8, 3.4))
    plt.plot(*uvM.T, color='#E08A2E', lw=3)
    for a, b in E:
        plt.plot(*uv[[a, b]].T, color='#3b6fb5', lw=2)
    plt.title('AR: куб поверх маркера (задание 3)')
    plt.gca().set_aspect('equal'); plt.xlim(0, 1280); plt.ylim(720, 0)
    plt.tight_layout(); plt.show()
    print()
    print('Вывод: 81 угол маркера дал гомографию, из неё — позу камеры '
          '(совпадение с точной до %.0e), и куб с вашим наклоном %.2f рад просто '
          'проецируется этой позой в кадр. Наклон — переменный контент, трекинг '
          'от него не зависит: AR = гомография + проекция.'
          % (np.abs(R_est - R0).max(), th))

task_block('Задание 3. Поза из гомографии и виртуальный объект',
           'поза + наложение куба; здесь именно тот куб, что в лабе',
           _w, _lab4_t3)

# Задание 4: уточнение позы Гауссом-Ньютоном
_w = [W.FloatSlider(value=0.5, min=0, max=2, step=0.05, description='шум, пикс'),
      W.FloatSlider(value=0.15, min=0, max=0.5, step=0.01, description='старт. ошибка')]

def _lab4_t4(_w=_w):
    s, e0 = _w[0].value, _w[1].value
    r = np.random.default_rng(0)
    obs = obs_clean + r.normal(0, s, obs_clean.shape)
    xi_true = np.r_[so3_log(R0), T0]
    xi0 = xi_true + np.array([e0, 0.2 * e0, -e0, 0.1, -0.1, e0 * 0.5])
    xi, hist = pnp_gauss_newton(Xw, obs, xi0)
    floor = s * np.sqrt(2 * len(obs))          # невязку ниже шума не опустить
    print('Гаусс-Ньютон PnP: невязка %.1f -> %.2e за %d итераций'
          % (hist[0], hist[-1], len(hist) - 1))
    print('пол невязки (уровень шума на %d наблюдений): ~ %.2e' % (2 * len(obs), floor))
    if e0 < 1e-9:
        print('старт уже в решении: невязка всю дорога на полу шума')
        ok(True, 'итерации не уходят ниже уровня шума (это физический пол)')
    else:
        ok(hist[-1] < 3 * floor + 1e-6,
           'метод дошёл до пола шума (шум ставит предел точности)')
        if len(hist) > 5 and hist[-3] > floor:
            print('до пола сходится квадратично (r убывает на порядки за шаг), '
                  'потом флуктуирует вокруг минимума')
    plt.semilogy(hist, 'o-')
    plt.axhline(floor, color='0.6', ls='--', label='пол = уровень шума')
    plt.legend()
    plt.xlabel('итерация'); plt.ylabel('||невязка||')
    plt.title('ЛР_04 задание 4: убывание невязки Гаусса-Ньютона')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: старт с ошибкой %.2f при шуме %.1f пикс: невязка упала с %.1f до '
          '%.2e за %d итераций и легла на пол %.1e — это сам шум наблюдений. '
          'Быстрее пола сойтись нельзя: улучшать надо наблюдения, а не итерации. '
          'Крутизна первых шагов — квадратичная сходимость Ньютона.'
          % (e0, s, hist[0], hist[-1], len(hist) - 1, floor))

task_block('Задание 4. Уточнение позы методом Гаусса—Ньютона',
           'график невязки по итерациям: пол = уровень шума', _w, _lab4_t4)

# Задание 5: RANSAC против выбросов
_w = [W.FloatSlider(value=0.3, min=0, max=0.6, step=0.05, description='доля выбросов')]

def _lab4_t5(_w=_w):
    frac = _w[0].value
    r = np.random.default_rng(0)
    obs = obs_clean + r.normal(0, 0.3, obs_clean.shape)
    nbad = int(frac * len(obs))
    inl_true = np.ones(len(obs), bool)
    if nbad:
        bad = r.choice(len(obs), nbad, replace=False)
        obs[bad] += r.uniform(-150, 150, (nbad, 2))
        inl_true[bad] = False
    Hb = dlt_normalized(M2, obs)
    Hr, inl = ransac_homography(M2, obs, seed=0)
    t_b = np.linalg.norm(pose_from_homography(Hb, K)[1] - (R0 @ MARKER_C + T0))
    t_r = np.linalg.norm(pose_from_homography(Hr, K)[1] - (R0 @ MARKER_C + T0))
    print('ошибка позы: без защиты %.3f м, с RANSAC %.3f м' % (t_b, t_r))
    print('inliers: %d (из них ложных %d из %d подмешанных)'
          % (inl.sum(), (inl & ~inl_true).sum(), nbad))
    ok(t_r < 0.15 and (inl & ~inl_true).sum() <= 2,
       'RANSAC отбивает выбросы и переоценивает по inliers')
    improvement = t_b / max(t_r, 1e-9)
    print()
    print('Вывод: при доле мусора %.0f%% без защиты ошибка позы %.3f м, с RANSAC — '
          '%.3f м (лучше в %.0f раз), ложных инлаеров %d. МНК тащит за собой '
          'испорченные точки, RANSAC ищет консенсус и переоценивает по чистым — '
          'то же происходит в реальном трекинге с бликами и наложениями.'
          % (100 * frac, t_b, t_r, improvement, (inl & ~inl_true).sum()))

task_block('Задание 5. RANSAC против выбросов',
           'один и тот же DLT: с RANSAC и без; доля мусора регулируется',
           _w, _lab4_t5)

# Задание 6: сколько итераций нужно
_w = [W.FloatSlider(value=0.3, min=0.05, max=0.7, step=0.05, description='доля выбросов eps'),
      W.FloatSlider(value=0.99, min=0.5, max=0.9999, step=0.0001, description='вероятность p')]

def _lab4_t6(_w=_w):
    eps, p = _w[0].value, _w[1].value
    s = 4.0                                    # минимальная выборка: гомография
    q = (1 - eps) ** s                         # вероятность чистой четвёрки
    n_th = np.log(1 - p) / np.log(1 - q)
    print('вероятность чистой четвёрки: q = (1-eps)^4 = %.4f' % q)
    print('ожидание числа попыток до первой чистой: 1/q = %.1f' % (1 / q))
    print('формула RANSAC (чтобы с вероятностью p хотя бы одна была чистой):')
    print('  N >= log(1-p)/log(1-q) = %.1f итераций' % n_th)
    r = np.random.default_rng(0)
    trials = []
    for _ in range(200):
        k = 0
        while True:
            k += 1
            if r.random() < q:
                break
            if k > 100000:
                break
        trials.append(k)
    trials = np.array(trials, float)
    print('эксперимент (200 прогонов): среднее %.1f, медиана %.0f итераций'
          % (trials.mean(), np.median(trials)))
    ok(abs(trials.mean() - 1 / q) * q < 0.2,
       'среднее число попыток = 1/q (геометрическое распределение)')
    ok(np.median(trials) <= n_th * 1.5,
       'медиана укладывается в оценку формулы (формула - про надёжность, не про среднее)')
    print()
    print('Вывод: eps=%.2f -> q=%.4f: первая чистая четвёрка в среднем за %.1f '
          'попытки (1/q), а формула требует N=%.0f для надёжности p=%.2f — это '
          'разные характеристики (среднее против квантиля), и обе подтверждаются '
          'вашим Монте-Карло из %d прогонов.'
          % (eps, q, 1 / q, n_th, p, len(trials)))

task_block('Задание 6. Сколько итераций нужно на самом деле',
           'формула RANSAC против Монте-Карло', _w, _lab4_t6)

## ЛР_05 · Конфигурационное пространство и прямая задача кинематики

In [6]:
# ---------- ЛР_05, задания 1-7 (интерактивно) ----------
fig_sentinel = None

# Задание 1: матрица DH
_w = [W.FloatSlider(value=0.5, min=0, max=1.5, step=0.05, description='a, м'),
      W.FloatSlider(value=0.0, min=-1.57, max=1.57, step=0.05, description='alpha, рад'),
      W.FloatSlider(value=0.3, min=0, max=1, step=0.05, description='d, м'),
      W.FloatSlider(value=0.6, min=-3.14, max=3.14, step=0.05, description='theta, рад')]

def _lab5_t1(_w=_w):
    a, al, d, th = (s.value for s in _w)
    T = dh(a, al, d, th)
    print('DH(a=%.2f, alpha=%.2f, d=%.2f, theta=%.2f) =\n' % (a, al, d, th), T)
    R = T[:3, :3]
    ok(np.abs(R.T @ R - np.eye(3)).max() < 1e-12, 'R^T R = I')
    ok(abs(np.linalg.det(R) - 1) < 1e-12, 'det R = +1')
    print()
    print('Вывод: ваши четыре параметра (a=%.2f, alpha=%.2f, d=%.2f, theta=%.2f) '
          'упаковали в одну матрицу четыре движения (два поворота + два '
          'переноса), и R^TR=I, det=+1 при любых значениях. Каждое звено любой '
          'реальной руки строится этой же матрицей: меняете 4 параметра — '
          'пересобираете весь робот.'
          % (a, al, d, th))

task_block('Задание 1. Матрица Денавита—Хартенберга',
           'живая матрица из четырёх параметров + проверки', _w, _lab5_t1)

# Задание 2: плоский 3R
_w = [W.FloatSlider(value=0.5, min=-3.14, max=3.14, step=0.02, description='q1'),
      W.FloatSlider(value=0.4, min=-3.14, max=3.14, step=0.02, description='q2'),
      W.FloatSlider(value=-0.3, min=-3.14, max=3.14, step=0.02, description='q3')]

def _lab5_t2(_w=_w):
    q = np.array([s.value for s in _w])
    pts = [np.zeros(2)]
    T = np.eye(4)
    for a, th in zip(L3_ARM, q):
        T = T @ dh(a, 0, 0, th)
        pts.append(T[:2, 3].copy())
    pts = np.array(pts)
    p_fk = fk3_planar(q)
    ok(np.abs(pts[-1] - p_fk).max() < 1e-12,
       'DH-произведение == замкнутая форма с суммами углов')
    diff = float(np.abs(pts[-1] - p_fk).max())
    ok(diff < 1e-12,
       'DH-произведение == замкнутая форма с суммами углов')
    plt.figure(figsize=(3.8, 3.6))
    plt.plot(pts[:, 0], pts[:, 1], 'o-', lw=3, color='#1F2733')
    plt.plot(*p_fk, 'x', color='#d64545', ms=12)
    plt.axis('equal'); plt.grid(True)
    plt.title('3R: схват по DH, крест - замкнутая форма')
    plt.tight_layout(); plt.show()
    print()
    print('Вывод: q=(%.2f, %.2f, %.2f) -> схват (%.3f, %.3f) м; произведение '
          'DH-матриц и замкнутая формула с накоплением углов совпали до %.0e. '
          'Символьный вывод из лабы проверяется этим же числом: матричная цепочка '
          'и аналитика — одно и то же.'
          % (q[0], q[1], q[2], p_fk[0], p_fk[1], max(diff, 2.2e-16)))

task_block('Задание 2. Плоский 3R (у лабы - символьно)',
           'числовой прогон той же цепочки; символьный вывод - в решении ЛР_05',
           _w, _lab5_t2)

# Задание 3: символьная компиляция (аналог UR5/lambdify)
_w = [W.IntSlider(value=20000, min=2000, max=100000, step=2000, description='вызовов N')]

def _lab5_t3(_w=_w):
    import sympy as sp
    import time
    q1s, q2s, q3s = sp.symbols('q1 q2 q3')
    l1, l2, l3 = [sp.Float(v) for v in L3_ARM]
    x = (l1 * sp.cos(q1s) + l2 * sp.cos(q1s + q2s)
         + l3 * sp.cos(q1s + q2s + q3s))
    y = (l1 * sp.sin(q1s) + l2 * sp.sin(q1s + q2s)
         + l3 * sp.sin(q1s + q2s + q3s))
    fx = sp.lambdify((q1s, q2s, q3s), x, 'numpy')
    fy = sp.lambdify((q1s, q2s, q3s), y, 'numpy')
    N = _w[0].value
    qs = np.random.default_rng(0).uniform(-3, 3, (N, 3))
    t0 = time.perf_counter()
    Xc = fx(qs[:, 0], qs[:, 1], qs[:, 2])
    Yc = fy(qs[:, 0], qs[:, 1], qs[:, 2])
    t_c = (time.perf_counter() - t0) / N * 1e6
    t0 = time.perf_counter()
    a1 = qs[:, 0]; a2 = a1 + qs[:, 1]; a3 = a2 + qs[:, 2]
    Xn = (L3_ARM[0] * np.cos(a1) + L3_ARM[1] * np.cos(a2)
          + L3_ARM[2] * np.cos(a3))
    t_n = (time.perf_counter() - t0) / N * 1e6
    ok(np.abs(Xc - Xn).max() < 1e-12, 'компилированная форма == исходной')
    print('компилированная (lambdify): %.2f мкс/вызов' % t_c)
    print('векторизованный NumPy:      %.2f мкс/вызов' % t_n)
    print('в лабе то же сравнение делается на UR5 (6 звеньев) - отношение то же')
    print()
    print('Вывод: N=%d вызовов: lambdify %.2f мкс, ручной NumPy %.2f мкс (отношение '
          '%.1f). Смысл компиляции не в скорости — векторизованный NumPy и так '
          'быстр — а в автогенерации кода из символьной модели: на UR5 с 6 '
          'звеньями формулу руками уже не выпишешь без ошибок. Значения совпали '
          'до %.0e.'
          % (N, t_c, t_n, t_n / max(t_c, 1e-12), np.abs(Xc - Xn).max()))

task_block('Задание 3. Компиляция символьной FK (аналог UR5/lambdify)',
           'cse + lambdify против NumPy: микросекунды на вызов', _w, _lab5_t3)

# Задание 4: рабочая зона и Монте-Карло
_w = [W.IntSlider(value=4000, min=500, max=20000, step=500, description='точек N')]

def _lab5_t4(_w=_w):
    N = _w[0].value
    n_plot = min(N, 6000)
    r = np.random.default_rng(0)
    qs = r.uniform(-np.pi, np.pi, (n_plot, 3))
    a1 = qs[:, 0]; a2 = a1 + qs[:, 1]; a3 = a2 + qs[:, 2]
    X = (L3_ARM[0] * np.cos(a1) + L3_ARM[1] * np.cos(a2) + L3_ARM[2] * np.cos(a3))
    Y = (L3_ARM[0] * np.sin(a1) + L3_ARM[1] * np.sin(a2) + L3_ARM[2] * np.sin(a3))
    P = np.column_stack([X, Y])
    r_out = float(L3_ARM.sum())
    R2 = X ** 2 + Y ** 2
    # сходимость оценки: доля конфигураций, чей схват в круге вылета
    ns = np.unique(np.logspace(2, np.log10(n_plot), 20).astype(int))
    fracs = np.array([np.mean(R2[:n] < r_out ** 2) for n in ns])
    err = np.abs(fracs - fracs[-1])
    plt.figure(figsize=(6.6, 3.4))
    plt.subplot(1, 2, 1)
    plt.scatter(P[:, 0], P[:, 1], s=2, alpha=0.3, color='#3b6fb5')
    th = np.linspace(0, 2 * np.pi, 100)
    plt.plot(r_out * np.cos(th), r_out * np.sin(th), color='#d64545', lw=1.2)
    plt.axis('equal'); plt.title('облако рабочей зоны 3R (Монте-Карло)')
    plt.subplot(1, 2, 2)
    plt.loglog(ns, np.maximum(err, 1e-6), 'o-')
    k = err[np.argmax(err > 0)] * np.sqrt(ns[np.argmax(err > 0)])         if (err > 0).any() else 1e-3
    plt.plot(ns, k / np.sqrt(ns), '--', label='C/sqrt(N)')
    plt.legend(); plt.grid(True); plt.title('сходимость оценки')
    plt.xlabel('N'); plt.ylabel('|frac - frac_final|')
    plt.tight_layout(); plt.show()
    print('доля конфигураций в круге вылета: %.4f -> площадь ~ %.4f м^2 '
          '(полный круг %.4f м^2), сходимость ~ 1/sqrt(N)'
          % (fracs[-1], np.pi * r_out ** 2 * fracs[-1], np.pi * r_out ** 2))
    print()
    print('Вывод: при N=%d доля схватов в круге вылета %.3f -> оценка площади '
          '%.3f м^2 из πR^2=%.3f м^2: 3R не просто дотягивается до круга, а '
          'перекрывает его с запасом конфигураций (одна точка — много поз). '
          'Правый график: разброс оценки падает как 1/√N — тяните N и следите.'
          % (N, fracs[-1], np.pi * r_out ** 2 * fracs[-1], np.pi * r_out ** 2))

task_block('Задание 4. Рабочая зона и сходимость Монте-Карло',
           'N регулируется: облако + сходимость оценки площади', _w, _lab5_t4)

# Задание 5: объём рабочей зоны
_w = [W.IntSlider(value=20000, min=2000, max=60000, step=2000, description='точек N')]

def _lab5_t5(_w=_w):
    N = _w[0].value
    r = np.random.default_rng(0)
    qq = r.uniform(-np.pi, np.pi, (N, 3))
    rr = np.array([np.sum(fk3_planar(x) ** 2) for x in qq])
    r_out = L3_ARM.sum()
    frac = np.mean(rr < r_out ** 2)
    est = frac * np.pi * r_out ** 2
    bias_dir = 'вниз' if frac < 0.5 else 'вверх'
    print('доля достижимости по конфигурациям: %.4f -> площадь %.4f м^2' % (frac, est))
    print('оценка Монте-Карло смещена %s при росте N смещение уходит, СКО ~ 1/sqrt(N)'
          % bias_dir)
    ok(est > 0.5 * np.pi * r_out ** 2, 'планарный 3R покрывает большую часть круга вылета')
    print()
    print('Вывод: N=%d проб: доля %.4f -> площадь %.4f м^2 против полного круга '
          '%.4f м^2 (%.0f%%). Рост N меняет оценку слабо, а разброс падает как '
          '1/√N — в лабе этим же методом оценивают уже объём 3D-рабочей зоны '
          'реального манипулятора.'
          % (N, frac, est, np.pi * r_out ** 2,
             100 * est / (np.pi * r_out ** 2)))

task_block('Задание 5. Объём рабочей зоны (метод Монте-Карло)',
           'плоская версия метода лабы: доля достижимых конфигураций',
           _w, _lab5_t5)

# Задание 6: PoE == DH
_w = [W.FloatSlider(value=0.7, min=-3, max=3, step=0.02, description='q1'),
      W.FloatSlider(value=-0.4, min=-3, max=3, step=0.02, description='q2'),
      W.FloatSlider(value=0.9, min=-3, max=3, step=0.02, description='q3')]

def _lab5_t6(_w=_w):
    q = np.array([s.value for s in _w])
    T = np.eye(4)
    for a, th in zip(L3_ARM, q):
        T = T @ dh(a, 0, 0, th)
    p_dh = T[:2, 3]
    # произведение экспонент (Modern Robotics, пространственные винты):
    # оси z через положения шарниров в «домашней» (нулевой) конфигурации
    p_j = [np.zeros(3),
           np.array([L3_ARM[0], 0.0, 0.0]),
           np.array([L3_ARM[0] + L3_ARM[1], 0.0, 0.0])]
    M = np.eye(4)                      # домашняя поза схвата: рука вдоль x
    M[0, 3] = float(L3_ARM.sum())
    Te = np.eye(4)
    for th, pj in zip(q, p_j):
        R = Rz(th)                     # вращение вокруг оси z, проходящей через pj
        Ti = np.eye(4)
        Ti[:3, :3] = R
        Ti[:3, 3] = pj - R @ pj
        Te = Te @ Ti
    T_poe = Te @ M
    p_poe = T_poe[:2, 3]
    print('DH:  схват = %s' % p_dh.round(6))
    print('PoE: схват = %s' % p_poe.round(6))
    ok(np.abs(p_poe - p_dh).max() < 1e-10, 'произведение экспонент == DH')
    print()
    print('Вывод: при q=(%.2f, %.2f, %.2f) обе параметризации привели схват в одну '
          'точку (расхождение %.0e): DH описывает цепочку локальными кадрами, '
          'PoE — винтами из «домашней» позы. Эквивалентны оба; PoE — язык '
          'современных солверов (Modern Robotics), на котором выводят якобианы и '
          'динамику без промежуточных кадров.'
          % (q[0], q[1], q[2], max(np.abs(p_poe - p_dh).max(), 2.2e-16)))

task_block('Задание 6. Произведение экспонент и его эквивалентность DH',
           'одна конфигурация, две параметризации - совпадают', _w, _lab5_t6)

# Задание 7: движение по последовательности конфигураций
_w = [W.IntSlider(value=12, min=2, max=40, step=1, description='точек пути')]

def _lab5_t7(_w=_w):
    n = _w[0].value
    qa, qb = np.array([-0.5, 0.2, 0.8]), np.array([1.8, -0.9, -0.4])
    ts = np.linspace(0, 1, n)
    plt.figure(figsize=(5.2, 3.6))
    for t in ts:
        q = qa + t * (qb - qa)
        pts = [np.zeros(2)]
        T = np.eye(4)
        for a, th in zip(L3_ARM, q):
            T = T @ dh(a, 0, 0, th)
            pts.append(T[:2, 3].copy())
        pts = np.array(pts)
        plt.plot(pts[:, 0], pts[:, 1], 'o-', lw=1.2, alpha=0.5, color='#3b6fb5')
    path = np.array([fk3_planar(qa + t * (qb - qa)) for t in ts])
    plt.plot(path[:, 0], path[:, 1], 'r-', lw=2, label='схват')
    plt.axis('equal'); plt.grid(True); plt.legend()
    plt.title('последовательность конфигураций -> траектория схвата')
    plt.tight_layout(); plt.show()
    ok(np.linalg.norm(path[-1] - fk3_planar(qb)) < 1e-12,
       'конец пути == FK конечной конфигурации')
    seg = np.linalg.norm(np.diff(path, axis=0), axis=1)
    print()
    print('Вывод: %d узлов интерполяции: углы меняются равномерно, но длина дуги '
          'схвата по сегментам гуляет от %.3f до %.3f м — скорость схвата зависит '
          'от конфигурации (это якобиан, ЛР_06). Для равномерной скорости '
          'интерполируют по дуге, а не по углам.'
          % (n, seg.min(), seg.max()))

task_block('Задание 7. Движение по последовательности конфигураций',
           'интерполяция углов -> веер поз + траектория схвата', _w, _lab5_t7)

## ЛР_06 · Обратная задача кинематики

In [7]:
# ---------- ЛР_06, задания 1-7 (интерактивно) ----------

# Задание 1: FK и якобиан 2R
_w = [W.FloatSlider(value=0.6, min=-3.14, max=3.14, step=0.02, description='q1'),
      W.FloatSlider(value=0.8, min=-3.14, max=3.14, step=0.02, description='q2')]

def _lab6_t1(_w=_w):
    q = np.array([s.value for s in _w])
    J = jac2(q)
    h = 1e-6
    Jn = np.column_stack([(fk2(q + h * e) - fk2(q - h * e)) / (2 * h)
                          for e in np.eye(2)])
    det_analytic = L2_1 * L2_2 * np.sin(q[1])
    print('схват: %s м' % fk2(q).round(4))
    print('J =\n', J)
    ok(np.abs(J - Jn).max() < 1e-8, 'аналитический якобиан == численному')
    ok(abs(np.linalg.det(J) - det_analytic) < 1e-9,
       'det J = l1*l2*sin(q2) = %.4f' % det_analytic)
    print('сингулярность при q2 -> 0 или pi (рука вытянута/сложена)')
    print()
    print('Вывод: q2=%.2f рад -> det J = l1*l2*sin(q2) = %.4f, схват в (%.3f, '
          '%.3f). Тяните q2 к 0 или pi: det, а за ним cond(J), обвалятся — рука '
          'теряет поперечную подвижность, и ньютоновский IK (задание 3) '
          'начнёт захлёбываться именно у этих конфигураций.'
          % (q[1], det_analytic, fk2(q)[0], fk2(q)[1]))

task_block('Задание 1. Прямая кинематика и якобиан (2R)',
           'якобиан + det; вырождение видно сразу', _w, _lab6_t1)

# Задание 2: аналитическое решение - две ветви
_w = [W.FloatSlider(value=0.6, min=-1.2, max=1.2, step=0.02, description='цель x'),
      W.FloatSlider(value=0.4, min=-1.2, max=1.2, step=0.02, description='цель y')]

def _lab6_t2(_w=_w):
    target = np.array([s.value for s in _w])
    sols = ik2_analytic(target)
    rlim = L2_1 + L2_2
    plt.figure(figsize=(3.8, 3.6))
    th = np.linspace(0, 2 * np.pi, 100)
    plt.plot(rlim * np.cos(th), rlim * np.sin(th), '--', color='0.8')
    if sols is None:
        print('цель %s вне рабочей зоны (вылет %.2f > l1+l2 = %.2f) - решений нет'
              % (target.round(3), np.linalg.norm(target), rlim))
    else:
        colors = ['#d64545', '#3b6fb5']
        for q, c in zip(sols, colors):
            pts = np.array([[0, 0], [L2_1 * np.cos(q[0]), L2_1 * np.sin(q[0])], fk2(q)])
            plt.plot(pts[:, 0], pts[:, 1], 'o-', color=c, lw=2.5,
                     label='q = (%.2f, %.2f)' % tuple(q))
        plt.legend()
        ok(abs(np.linalg.norm(fk2(sols[0]) - target)) < 1e-9
           and abs(np.linalg.norm(fk2(sols[1]) - target)) < 1e-9,
           'обе ветви попадают ровно в цель')
    plt.plot(*target, 'x', color='k', ms=12)
    plt.axis('equal'); plt.grid(True)
    plt.title('две ветви аналитического IK')
    plt.tight_layout(); plt.show()
    r_t = np.linalg.norm(target)
    print()
    if sols is None:
        print('Вывод: цель (%.2f, %.2f), вылет %.3f м — вне кольца %.2f..%.2f м: '
              'геометрически решений нет, любой итеративный солвер лишь подберётся '
              'к границе максимально близко — про это задание 6.'
              % (target[0], target[1], r_t, abs(L2_1 - L2_2), L2_1 + L2_2))
    else:
        print('Вывод: цель (%.2f, %.2f) при вылете %.3f м попадает в кольцо '
              '%.2f..%.2f м — ровно две ветви: «локоть вверх» q=(%.2f, %.2f) и '
              '«локоть вниз» q=(%.2f, %.2f). Выбор между ними — не математика, а '
              'условия задачи: препятствия, стартовая поза, энергия.'
              % (target[0], target[1], r_t, abs(L2_1 - L2_2), L2_1 + L2_2,
                 sols[0][0], sols[0][1], sols[1][0], sols[1][1]))

task_block('Задание 2. Аналитическое решение: две ветви',
           'внутри кольца - ровно два решения; выведите цель за кольцо',
           _w, _lab6_t2)

# Задание 3: Ньютон и скорость сходимости
_w = [W.FloatSlider(value=0.6, min=-1.2, max=1.2, step=0.02, description='цель x'),
      W.FloatSlider(value=0.4, min=-1.2, max=1.2, step=0.02, description='цель y'),
      W.FloatSlider(value=0.5, min=-3, max=3, step=0.05, description='семя q1'),
      W.FloatSlider(value=0.5, min=-3, max=3, step=0.05, description='семя q2')]

def _lab6_t3(_w=_w):
    target = np.array([_w[0].value, _w[1].value])
    q0 = [_w[2].value, _w[3].value]
    q, hist = ik2_newton(target, q0)
    sols = ik2_analytic(target)
    if sols is None:
        print('цель (%.2f, %.2f): %s м вне рабочей зоны (кольцо %.2f..%.2f м) - '
              'аналитического решения нет'
              % (target[0], target[1], np.linalg.norm(target),
                 abs(L2_1 - L2_2), L2_1 + L2_2))
        return
    dists = [abs(q[1] - s[1]) for s in sols]
    sols_sorted = [s for _, s in sorted(zip(dists, sols))]
    knee = 'локоть вверх' if sols_sorted[0][1] > 0 else 'локоть вниз'
    print('Ньютон: %d итераций, невязка %.2e -> ветвь «%s»'
          % (len(hist) - 1, hist[-1], knee))
    if len(hist) > 4:
        rat = hist[-1] / hist[-3]
        print('скорость: r_n/r_(n-2) = %.1e -> сходимость квадратичная' % rat)
        ok(hist[-1] < 1e-9, 'сошёлся')
        ok(rat < 1e-3, 'знаменатель сжимается квадратично (до пола шума)')
    plt.semilogy(hist, 'o-')
    plt.xlabel('итерация'); plt.ylabel('||fk(q)-target||')
    plt.title('ЛР_06 задание 3: сходимость Ньютона (pinv)')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: семя (%.2f, %.2f), цель (%.2f, %.2f): сошлось за %d итераций в '
          'ветвь «%s», невязка упала с %.2e до %.2e. Отношения соседних хвостов '
          'уходят в ноль на глазах — каждая итерация возводит остаток примерно в '
          'квадрат (квадратичная сходимость). Смените семя на противоположное — '
          'попадёте в другую ветвь (задание 4).'
          % (q0[0], q0[1], target[0], target[1], len(hist) - 1, knee,
             hist[0], hist[-1]))

task_block('Задание 3. Метод Ньютона и скорость сходимости',
           'семя решает, в какую ветвь попадёт; график невязки - квадратичный',
           _w, _lab6_t3)

# Задание 4: бассейны притяжения
_w = [W.IntSlider(value=40, min=10, max=60, step=5, description='сетка n'),
      W.FloatSlider(value=0.6, min=0.2, max=0.9, step=0.02, description='цель x')]

def _lab6_t4(_w=_w):
    n = _w[0].value
    target = np.array([_w[1].value, 0.35])
    qs1 = np.linspace(-np.pi, np.pi, n)
    qs2 = np.linspace(-np.pi, np.pi, n)
    it_map = np.zeros((n, n))
    br_map = np.zeros((n, n))
    sols = ik2_analytic(target) or [np.array([9, 9])] * 2
    for i, a in enumerate(qs1):
        for j, b in enumerate(qs2):
            q, hist = ik2_newton(target, [a, b], iters=30)
            it_map[j, i] = len(hist)
            br_map[j, i] = 0 if abs(q[1] - sols[0][1]) < abs(q[1] - sols[1][1]) else 1
    plt.figure(figsize=(7.6, 3.2))
    plt.subplot(1, 2, 1)
    plt.imshow(br_map, extent=[-np.pi, np.pi, -np.pi, np.pi], origin='lower',
               cmap='coolwarm', aspect='auto')
    plt.colorbar(label='ветвь'); plt.xlabel('семя q1'); plt.ylabel('семя q2')
    plt.title('бассейны: какая ветвь поймана')
    plt.subplot(1, 2, 2)
    plt.imshow(it_map, extent=[-np.pi, np.pi, -np.pi, np.pi], origin='lower',
               cmap='viridis', aspect='auto')
    plt.colorbar(label='итераций')
    plt.xlabel('семя q1')
    plt.title('итерации до сходимости')
    plt.tight_layout(); plt.show()
    print('граница бассейнов изрезанная - так и должно быть (задание 4)')
    f0 = float(np.mean(br_map == 0))
    hard = int(np.sum(it_map >= 30))
    print()
    print('Вывод: сетка %dx%d ваших семян: %.0f%% сошлись в ветвь-0, %.0f%% в '
          'ветвь-1, %d клеток упёрлись в лимит 30 итераций — они у границы '
          'бассейнов. Граница изрезана: сколь угодно близкие семена дают разные '
          'ветви, поэтому на практике солверу подают осмысленное семя — текущую '
          'позу робота, а не случайное число.'
          % (n, n, 100 * f0, 100 * (1 - f0), hard))

task_block('Задание 4. Бассейны притяжения',
           'карта по сетке семян: ветвь + количество итераций', _w, _lab6_t4)

# Задание 5: у границы рабочей зоны
_w = [W.FloatLogSlider(value=0.01, base=10, min=-4, max=-0.5, step=0.1,
                       description='отступ d')]

def _lab6_t5(_w=_w):
    d = _w[0].value
    r_out = L2_1 + L2_2
    ds = np.logspace(-4, -0.5, 25)
    conds, hists = [], []
    for dd in ds:
        qs, hh = ik2_newton(np.array([r_out - dd, 0.02]),
                            [0.3, 0.1], iters=100)
        conds.append(np.linalg.cond(jac2(qs)))
        hists.append(hh[-1])
    q, hist = ik2_newton(np.array([r_out - d, 0.02]), [0.3, 0.1], iters=100)
    print('d = %.0e: cond(J) в решении = %.0f, невязка = %.1e'
          % (d, np.linalg.cond(jac2(q)), hist[-1]))
    slope = np.polyfit(np.log(ds), np.log(conds), 1)[0]
    print('наклон cond(J) ~ d^(%.2f) (теория у вытянутой руки: d^-1/2 -> -0.5)'
          % slope)
    ok(abs(slope + 0.5) < 0.25, 'степенной закон согласуется')
    plt.loglog(ds, conds, 'o-')
    plt.xlabel('отступ от границы d, м'); plt.ylabel('cond(J)')
    plt.title('ЛР_06 задание 5: вырождение якобиана у границы')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: при вашем d=%.0e м cond(J)=%.0f; по всей кривой отступов '
          'cond растёт как d^(%.2f) — степенной закон с показателем около -1/2 '
          '(теория вытянутой руки). Ньютон ещё сходится (невязка %.1e), но '
          'дорожает по итерациям; у границы выручает демпфирование — задание 6.'
          % (d, np.linalg.cond(jac2(q)), slope, hist[-1]))

task_block('Задание 5. Что происходит вблизи границы рабочей зоны',
           'cond(J) против отступа: лог-лог и степень -1/2', _w, _lab6_t5)

# Задание 6: демпфированный солвер на недостижимой цели
_w = [W.FloatLogSlider(value=0.1, base=10, min=-2, max=0, step=0.25,
                       description='лямбда')]

def _lab6_t6(_w=_w):
    lam = _w[0].value
    target_bad = np.array([L2_1 + L2_2 + 0.15, 0.15])     # за границей зоны
    gap = np.linalg.norm(target_bad) - (L2_1 + L2_2)       # минимально возможная невязка
    qs, hs = ik2_newton(target_bad, [0.3, 0.2], damp=lam, iters=40)
    q0, h0 = ik2_newton(target_bad, [0.3, 0.2], damp=0.0, iters=40)
    print('цель за границей зоны: %.3f м от оси (макс %.2f), '
          'недобор %.2f м' % (np.linalg.norm(target_bad), L2_1 + L2_2, gap))
    print('чистый Ньютон:      невязка %.4f (проскакивает границу туда-сюда)'
          % h0[-1])
    print('DLS лямбда=%.0e:    невязка %.4f' % (lam, hs[-1]))
    ok(np.isfinite(hs).all(), 'демпфированный солвер не расходится (итерации конечны)')
    if hs[-1] < gap * 2.5 + 0.05:
        ok(True, 'DLS останавливается у границы с минимальной невязкой (~%.2f)' % gap)
    else:
        print('  ...  лямбда=%.0e ещё мала: шаг проскакивает оптимум - '
              'увеличьте демпфирование' % lam)
    plt.semilogy(h0, label='чистый Ньютон')
    plt.semilogy(hs, label='DLS, лямбда=%.0e' % lam)
    plt.axhline(gap, color='0.6', ls='--', label='теоретический минимум')
    plt.xlabel('итерация'); plt.ylabel('невязка'); plt.legend(); plt.grid(True)
    plt.title('ЛР_06 задание 6: недостижимая цель, DLS против чистого Ньютона')
    plt.tight_layout(); plt.show()
    print()
    if hs[-1] < gap * 2.5 + 0.05:
        print('Вывод: λ=%.0e: невязка %.3f при теоретическом минимуме %.3f — '
              'солвер остановился у границы честно, не разлетевшись. Плата за '
              'устойчивость: сходимость из квадратичной становится линейной — '
              'тот компромисс, что стоит в Trac-IK/KDL.'
              % (lam, hs[-1], gap))
    else:
        print('Вывод: λ=%.0e слишком мала: невязка %.3f проскакивает минимум '
              '%.3f (пунктир на графике). Поднимите λ — увидите, как солвер '
              'начинает останавливаться у границы вместо метаний чистого Ньютона'
              % (lam, hs[-1], gap))

task_block('Задание 6. Промышленный решатель с демпфированием',
           'Trac-IK/KDL-стиль: лямбда ограничивает шаг у сингулярности',
           _w, _lab6_t6)

# Задание 7: избыточный 3R и ядро якобиана
_w = [W.FloatSlider(value=0.7, min=-3, max=3, step=0.02, description='q1'),
      W.FloatSlider(value=-0.4, min=-3, max=3, step=0.02, description='q2'),
      W.FloatSlider(value=0.9, min=-3, max=3, step=0.02, description='q3')]

def _lab6_t7(_w=_w):
    q = np.array([s.value for s in _w])
    J = jac3_planar(q)
    ns = np.linalg.svd(J)[2][-1]
    ok(np.linalg.norm(J @ ns) < 1e-12,
       'J * ns = 0: движение вдоль ядра не двигает схват (1-й порядок)')
    ts = np.linspace(0, 0.2, 60)
    drift = np.array([np.linalg.norm(fk3_planar(q + t * ns) - fk3_planar(q))
                      for t in ts])
    print('ядро (SVD): ns = %s' % ns.round(3))
    print('сдвиг 0.10 рад вдоль ядра -> смещение схвата %.2e м '
          '(2-й порядок малости)' % drift[29])
    i1, i2 = 10, 40                       # t = 0.034 и 0.136: шаг x4
    growth = drift[i2] / max(drift[i1], 1e-18)
    ok(abs(growth - (ts[i2] / ts[i1]) ** 2) < 0.2 * growth,
       'смещение растёт как t^2 (квадрат шага)')
    plt.plot(ts * 100, drift * 1e4)
    plt.plot(ts * 100, drift[i2] * 1e4 / (ts[i2] / ts[i1]) ** 2 * (ts / ts[i1]) ** 2,
             '--', color='0.6', label='~ t^2')
    plt.legend()
    plt.xlabel('шаг вдоль ядра, сотые рад'); plt.ylabel('смещение схвата, 1e-4 м')
    plt.title('ЛР_06 задание 7: нуль-пространство - самовращение без движения схвата')
    plt.grid(True); plt.tight_layout(); plt.show()
    print()
    print('Вывод: при q=(%.2f, %.2f, %.2f) сдвиг на 0.10 рад вдоль ядра сместил '
          'схват всего на %.1e м — при точном J·ns=0 это чисто нелинейный остаток, '
          'растущий как t^2. Избыточность 3R не украшение: вдоль ядра уводят локоть '
          'от препятствий, не трогая положение схвата.'
          % (q[0], q[1], q[2], drift[29]))

task_block('Задание 7. Избыточный манипулятор: ядро якобиана',
           'SVD-ядро 3R: самовращение суставов при неподвижном схвате',
           _w, _lab6_t7)

## Сводная сцена · Рука, собранная из всех шести лаб

Финальная сцена работает как конвейер, где у каждого узла — табличка
с указанием лабы и задания. В 3D-виде подписи стоят прямо у узлов
(камера, маркер, база, схват, цель), а под графиком печатается
трассировка «РУКА СОБРАНА ИЗ ШЕСТИ ЛАБ»: каждая строка — формула
конкретного задания, работающая в этой сцене, с числом от ваших
ползунков. Двигаете ползунок — обновляются и подписи, и числа.

| Узел сцены | Откуда формула |
|---|---|
| Однородные 4×4 всех кадров, композиция | ЛР_01, задания 1–3 |
| Дерево кадров world → cam/marker/base → ee (tf2) | ЛР_01, задание 4 |
| Накопление ошибок при цепочках кадров | ЛР_01, задание 5 |
| SVD-проекция позы в SO(3) | ЛР_01, задание 7 |
| Перелёт камеры exp/log, кватернион (x,y,z,w) | ЛР_02, задания 1–2 |
| Путь перелёта по геодезической (slerp) | ЛР_02, задание 5 |
| Проекция K, стерео-глубина d = fB/Z | ЛР_03, задания 1, 4–5 |
| Дисторсия кадра (чекбокс) | ЛР_03, задание 3 |
| Гомография + нормировка Хартли + AR-поза | ЛР_04, задания 1–3 |
| RANSAC против выбросов | ЛР_04, задания 5–6 |
| Звенья руки, кадры суставов (DH/замкнутая форма), вылет | ЛР_05, задания 1–2, 4, 6 |
| Геометрический якобиан, IK Ньютоном, ядро (избыточность) | ЛР_06, задания 1, 3–5, 7 |

Рука — пространственная (yaw + 3 pitch, 4 шарнира, избыточность 1 DOF).

In [8]:
# Сводная панель (как прежде): режим, цель 3D, выбросы, перелёт камеры
mode = W.ToggleButtons(options=['ручной (суставы)', 'цель (IK)'],
                        description='Режим:')
q1 = W.FloatSlider(value=0.5, min=-np.pi, max=np.pi, step=0.01, description='q1 (yaw), рад')
q2 = W.FloatSlider(value=-0.5, min=-np.pi, max=np.pi, step=0.01, description='q2 (плечо)')
q3 = W.FloatSlider(value=0.9, min=-np.pi, max=np.pi, step=0.01, description='q3 (локоть)')
q4 = W.FloatSlider(value=0.6, min=-np.pi, max=np.pi, step=0.01, description='q4 (запястье)')
tx = W.FloatSlider(value=0.45, min=-0.9, max=0.9, step=0.01, description='цель x, м')
ty = W.FloatSlider(value=0.35, min=-0.9, max=0.9, step=0.01, description='цель y, м')
tz = W.FloatSlider(value=0.55, min=0.05, max=1.25, step=0.01, description='цель z, м')
out_frac = W.FloatSlider(value=0.2, min=0.0, max=0.6, step=0.05, description='выбросы')
seed_w = W.IntSlider(value=0, min=0, max=99, step=1, description='seed шума')
dist_on = W.Checkbox(False, description='дисторсия в кадре')
t_cam = W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01,
                      description='камера: перелёт A→B (ЛР_02)')

ui = W.VBox([mode,
             W.HTML('<b>Углы (в режиме IK — семя солвера)</b>'),
             W.HBox([q1, q2]), W.HBox([q3, q4]),
             W.HTML('<b>Цель схвата x/y/z (режим IK)</b>'),
             W.HBox([tx, ty, tz]),
             W.HTML('<b>Камера и кадр</b>'), W.HBox([t_cam, dist_on]),
             W.HBox([out_frac, seed_w])])
out_fig = W.Output()
out_txt = W.Output()
report = {}


def update(_=None):
    out_fig.clear_output(wait=True)
    out_txt.clear_output(wait=True)
    r = np.random.default_rng(seed_w.value)

    R_c, t_c, eye_c = cam_pose(t_cam.value)
    T_CAM_W = np.eye(4)
    T_CAM_W[:3, :3] = R_c.T
    T_CAM_W[:3, 3] = -R_c.T @ t_c
    q_cam = R_to_quat(R_c)

    q_seed = [q1.value, q2.value, q3.value, q4.value]
    ik_line = ''
    if mode.value.startswith('цель'):
        target = np.array([tx.value, ty.value, tz.value])
        q, hist, conds = ik3d_newton(target, q_seed)
        J = arm_jac(q)
        kernel_dim = 4 - np.linalg.matrix_rank(J, tol=1e-9)
        state = ('сошёлся' if hist[-1] < 1e-9 else
                 'НЕ ДОШЁЛ (цель вне зоны или у сингулярности)')
        ik_line = ('IK: итераций %d, невязка %.2e -> %s; cond(J) %.0f..%.0f, '
                   'ядро %d (избыточность)'
                   % (len(hist) - 1, hist[-1], state, conds.min(),
                      conds.max(), kernel_dim))
        report['ik_res'] = hist[-1]
    else:
        q = np.array(q_seed)
    points, frames, axes, origins = arm_fk(q)
    p_ee = points[-1]

    uvL, zL = project(K, R_c, t_c, MARKER_C[None])
    disp = F_PX * BASE_STEREO / zL[0]
    z_st = F_PX * BASE_STEREO / disp
    report['stereo_err'] = abs(z_st - zL[0])

    obs, _ = project(K, R_c, t_c, Xw)
    obs_n = obs + r.normal(0, 0.2, obs.shape)
    inl_true = np.ones(len(obs_n), bool)
    k = int(len(obs_n) * out_frac.value)
    if k:
        bad = r.choice(len(obs_n), k, replace=False)
        obs_n[bad] += r.uniform(-150, 150, (k, 2))
        inl_true[bad] = False
    Hh, inl = ransac_homography(M2, obs_n, seed=seed_w.value)
    R_est, t_est = pose_from_homography(Hh, K)
    dt = np.linalg.norm(t_est - (R_c @ MARKER_C + t_c))
    dang = np.degrees(np.arccos(np.clip((np.trace(R_est.T @ R_c) - 1) / 2, -1, 1)))
    report['pnp_dt'] = dt

    # ---- числа для трассировки «рука из шести лаб» ----
    # ЛР_01 з.4: tf2-lookup маркера через мир (два шага дерева, без прямой пары)
    T_wm = np.eye(4)
    T_wm[:3, 3] = MARKER_C
    err_chain = np.abs((np.linalg.inv(T_CAM_W) @ T_wm)[:3, 3]
                       - (R_c @ MARKER_C + t_c)).max()
    # ЛР_02 з.1/5: геодезическая между видами и отклонение перелёта от неё
    w_total = so3_log(R_A.T @ R_B)
    w_now = so3_log(R_A.T @ R_c)
    dev_geo = np.degrees(np.linalg.norm(w_now - t_cam.value * w_total))
    # ЛР_06 з.1/5/7: якобиан, его обусловленность и ядро
    J_arm = arm_jac(q)
    cond_J = np.linalg.cond(J_arm)
    ker_ns = np.linalg.svd(J_arm)[2][-1]
    ker_ns = ker_ns / np.linalg.norm(ker_ns)
    drift_ns = np.linalg.norm(arm_fk(q + 0.15 * ker_ns)[0][-1] - p_ee)
    reach_use = np.linalg.norm(p_ee - np.array([0.0, 0.0, D_BASE])) / REACH
    frames_ortho = max(np.abs(T3[:3, :3].T @ T3[:3, :3] - np.eye(3)).max()
                       for T3 in frames)
    # ЛР_04 з.6: сколько итераций RANSAC требует формула при вашей доле мусора
    q_clean = (1 - out_frac.value) ** 4
    n_ransac = 1.0 if q_clean > 1 - 1e-9 else np.log(1 - 0.999) / np.log(1 - q_clean)

    with out_fig:
        fig = plt.figure(figsize=(12.5, 4.6))
        ax1 = fig.add_subplot(1, 2, 1, projection='3d')
        for x in np.linspace(-0.6, 1.2, 7):
            ax1.plot([x, x], [-0.8, 1.2], [0, 0], color='0.85', lw=0.7)
        for y in np.linspace(-0.8, 1.2, 7):
            ax1.plot([-0.6, 1.2], [y, y], [0, 0], color='0.85', lw=0.7)
        MS = np.array([[-1, -1, 0], [1, -1, 0], [1, 1, 0], [-1, 1, 0], [-1, -1, 0]])
        MS = MS * MARKER_H + MARKER_C
        ax1.plot(*MS.T, color='#E08A2E', lw=3)
        ax1.plot(*points.T, color='#1F2733', lw=5, marker='o', ms=6)
        ax1.text(*p_ee, '  схват (FK: ЛР_05)', fontsize=7)
        ax1.text(*(eye_c + np.array([0.04, 0.0, 0.14])), 'камера (slerp: ЛР_02)',
                 fontsize=7, color='#3b6fb5')
        ax1.text(*(MARKER_C + np.array([0.02, -0.16, 0.08])),
                 'маркер (гомография+RANSAC: ЛР_04)', fontsize=7, color='#E08A2E')
        ax1.text(0.05, 0.0, D_BASE, '  база: DH-кадры (ЛР_05)',
                 fontsize=7, color='0.3')
        tgt3 = np.array([tx.value, ty.value, tz.value])
        ax1.scatter(*tgt3, marker='x', s=80, color='#b04848', depthshade=False)
        ax1.text(*(tgt3 + [0, 0, 0.07]), '  цель (IK: ЛР_06)',
                 fontsize=7, color='#b04848')
        ax1.plot(*zip(p_ee, tgt3), ls='--', lw=1.2, color='#b04848', alpha=0.7)
        draw_frame(ax1, np.eye(4), 'xyz')
        for T3 in frames:
            draw_frame(ax1, T3, 'xyz', L=0.07)
        draw_frame(ax1, T_CAM_W, 'XYZ')
        ax1.scatter(*eye_c, color='#3b6fb5', s=40)
        for pt in MS[::2]:
            ax1.plot(*zip(eye_c, pt), color='#3b6fb5', lw=0.6, ls='--', alpha=0.6)
        ax1.set_xlim(-0.6, 1.2); ax1.set_ylim(-1.0, 1.2); ax1.set_zlim(0, 2.0)
        ax1.set_box_aspect((1.8, 2.2, 2.0))
        ax1.set_title('Сцена: рука собрана из шести лаб - подписи у узлов')

        ax2 = fig.add_subplot(1, 2, 2)
        uvM, _ = project(K, R_c, t_c, MS)
        uvA, _ = project(K, R_c, t_c, points)
        if dist_on.value:
            uvM = distort_px(uvM); uvA = distort_px(uvA)
        ax2.plot(*uvM.T, color='#E08A2E', lw=2.5, label='маркер')
        ax2.plot(*uvA.T, color='#1F2733', lw=4, marker='o', ms=5, label='рука')
        uvT, _ = project(K, R_c, t_c, tgt3[None])
        if dist_on.value:
            uvT = distort_px(uvT)
        ax2.plot(*uvT[0], 'x', color='#d64545', ms=10, mew=2, label='цель')
        if k:
            ax2.plot(*obs_n[~inl_true].T, 'x', color='#d64545',
                     ms=4, alpha=0.4, label='выбросы')
        ax2.plot([CX], [CY], '+', color='#7a7a7a', ms=10)
        ax2.set_xlim(0, W_IMG); ax2.set_ylim(H_IMG, 0); ax2.set_aspect('equal')
        ax2.set_title('Кадр камеры (f=%.0f px)' % F_PX)
        ax2.legend(loc='lower right', fontsize=8)
        fig.suptitle('конвейер: пиксели (ЛР_03) -> поза маркера (ЛР_04) -> '
                     'дерево кадров (ЛР_01) -> перелёт (ЛР_02) -> FK руки (ЛР_05) -> IK (ЛР_06)',
                     fontsize=8)
        plt.tight_layout()
        fig.subplots_adjust(top=0.88)
        plt.show()

    with out_txt:
        print('дерево кадров (PARENT из ЛР_01): world -> {cam, marker, '
              'base, shoulder, elbow, wrist, ee}')
        print('камера: перелёт t=%.2f, кватернион (w,x,y,z) = %s  '
              '[(x,y,z,w) для ROS: %s]'
              % (t_cam.value, np.round(q_cam, 3),
                 np.round(q_cam[[1, 2, 3, 0]], 3)))
        print('рука:    q = %s рад' % np.round(q, 4))
        print('схват:   (%.3f, %.3f, %.3f) м, вылет от плеча %.3f из %.2f м'
              % (p_ee[0], p_ee[1], p_ee[2],
                 np.linalg.norm(p_ee - np.array([0, 0, D_BASE])), REACH))
        if ik_line:
            print(ik_line)
        print('стерео: диспаритет %.2f px -> глубина %.4f м (ошибка %.1e)'
              % (disp, z_st, report['stereo_err']))
        print('AR:     inliers верных %d/%d, ложных %d; ошибка позы %.4f м, %.2f град'
              % ((inl & inl_true).sum(), inl_true.sum(), (inl & ~inl_true).sum(),
                 dt, dang))
        print()
        print('===== РУКА СОБРАНА ИЗ ВСЕХ ШЕСТИ ЛАБ: где чья формула работает =====')
        print('[ЛР_01 з.1-3] каждый кадр сцены - однородная 4x4: у кадров руки '
              'max|R^T R - I| = %.0e, det = +1' % frames_ortho)
        print('[ЛР_01 з.4]  дерево кадров: lookup маркера в камере через мир, '
              'двумя шагами (как tf2 lookupTransform): невязка %.0e' % err_chain)
        print('[ЛР_01 з.5] цепочки кадров короткие (2 шага через мир): по заданию 5 '
              'ошибка растёт с каждым перемножением - дерево держат мелким')
        print('[ЛР_01 з.7] поза из H пришла неортогональной - SVD-шаг внутри '
              'pose_from_homography вернул её в SO(3), как в ортогонализации')
        print('[ЛР_02 з.1] угол между видами A и B: |log(Ra^T Rb)| = %.1f град '
              '(сама камера - его exp)' % np.degrees(np.linalg.norm(w_total)))
        print('[ЛР_02 з.2] та же ориентация кватернионом в ROS-порядке (x,y,z,w) '
              '= %s' % np.round(q_cam[[1, 2, 3, 0]], 3))
        print('[ЛР_02 з.5] перелёт t=%.2f идёт по геодезической: отклонение '
              'от неё %.0e град (slerp)' % (t_cam.value, max(dev_geo, 1e-15)))
        print('[ЛР_02 з.6] кадры пересобираются из углов заново каждый вызов - '
              'дрейфу интегрирования взяться неоткуда')
        print('[ЛР_03 з.1] проекция K: на глубине маркера %.2f м масштаб '
              'f/Z = %.0f пикс/м' % (zL[0], F_PX / zL[0]))
        print('[ЛР_03 з.4] стерео: d = fB/Z = %.2f пикс -> глубина %.4f м при '
              'базе %.2f м' % (disp, z_st, BASE_STEREO))
        print('[ЛР_03 з.5] цена пикселя шума на этой глубине: Z^2/(fB) = %.1e м - '
              'предел точности глубины' % (zL[0] ** 2 / (F_PX * BASE_STEREO)))
        print('[ЛР_03 з.3] дисторсия %s: у края кадра пиксели съезжают, как '
              'в лабе'
              % ('вкл - заметна по искривлённому контуру маркера на правой панели'
                 if dist_on.value else 'выкл (включите чекбокс - увидите бочку)'))
        print('[ЛР_04 з.1-2] гомография из 81 угла маркера: DLT + нормировка '
              'Хартли (cond плана ~1e6 -> ~20, как в задании 2)')
        print('[ЛР_04 з.3] AR-поза камеры над маркером из этой H: промах %.4f м, '
              '%.2f град' % (dt, dang))
        print('[ЛР_04 з.5] RANSAC при вашей доле мусора %.0f%%: верных инлаеров '
              '%d/%d, ложных %d' % (100 * out_frac.value, (inl & inl_true).sum(),
                                    inl_true.sum(), (inl & ~inl_true).sum()))
        print('[ЛР_04 з.6] формула требует N >= %.0f итераций (p=0.999, '
              '(1-eps)^4 = %.3f) - стенд держит запас: 2000'
              % (max(n_ransac, 1.0), q_clean))
        print('[ЛР_05 з.1-4] рука - DH-цепочка: звенья %.2f+%.2f+%.2f = вылет '
              '%.2f м; сейчас схват занял %.0f%% вылета'
              % (L_ARM[0], L_ARM[1], L_ARM[2], REACH, 100 * reach_use))
        print('[ЛР_05 з.6] та же FK - произведение экспонент (эквивалентно DH, '
              'проверено в задании)')
        if ik_line:
            print('[ЛР_06 з.3] IK Ньютоном: см. строку IK выше - семя это ваши '
                  'ползунки q1..q4')
            print('[ЛР_06 з.4] смените семя - получите другую ветвь решения '
                  '(у избыточной руки их континуум)')
            print('[ЛР_06 з.5] cond(J) вдоль итераций %.0f..%.0f - у границы '
                  'растёт, как в задании' % (conds.min(), conds.max()))
        else:
            print('[ЛР_06 з.3] включите режим «цель (IK)» - увидите ньютоновский '
                  'солвер в работе на этой же руке')
        print('[ЛР_06 з.7] ядро J: сдвиг 0.15 рад вдоль ns оставил схват на '
              'месте, сместив на %.1e м (2-й порядок) - избыточность в деле'
              % max(drift_ns, 1e-16))
        okk = (report['stereo_err'] < 1e-9
               and ('ik_res' not in report or report['ik_res'] < 1e-9)
               and report['pnp_dt'] < 0.15)
        print()
        print('СВОДНАЯ СЦЕНА: ' + ('ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ' if okk
                                   else 'расхождения - смотрите числа'))
        print()
        print('Вывод: камера t=%.2f, выбросы %.0f%%, дисторсия %s: стерео-глубина '
              '%.3f м (ошибка %.0e), AR-поза промахнулась на %.3f м (%.2f град) '
              '— конвейер «пиксели → поза → цель → суставы» работает%s. Ошибки '
              'каждого блока читаются в числах выше: шум ставит пол PnP, выбросы '
              'ловит RANSAC, недостижимость цели видит IK.'
              % (t_cam.value, 100 * out_frac.value,
                 'вкл' if dist_on.value else 'выкл',
                 z_st, report['stereo_err'], report['pnp_dt'], dang,
                 ' без сбоев' if okk
                 else ', но цель вне зоны руки — сдвиньте целевые ползунки'))

for c in [mode, q1, q2, q3, q4, out_frac, seed_w, dist_on, t_cam]:
    c.observe(update, names='value')


def _target_moved(_):
    if not mode.value.startswith('цель'):
        mode.value = 'цель (IK)'
    else:
        update()


tx.observe(_target_moved, names='value')
ty.observe(_target_moved, names='value')
tz.observe(_target_moved, names='value')
display(W.VBox([ui, out_fig, out_txt]))
update()

## Мост в ROS-стенд (опционально)

Если рядом поднят контейнер `robot-labs-stand` (см. README репозитория),
панель выше может управлять той же рукой уже в ROS 2: отмечаете галочку —
и каждое движение ползунков уходит мосту на TCP :8977, где
`bridge_server` превращает их в `/joint_states` и `/goal_pose`.
Рисуют руку rviz2 или Foxglove (WebSocket :9090). Без стенда ячейка
мирно печатает подсказку и ничего не ломает.

In [9]:
# Мост «ноутбук -> ROS-стенд»: TCP :8977, без зависимостей (socket, json)
import json as _json
import socket as _socket


def _ros_send(obj):
    """Одна попытка отправить сообщение мосту; None - мост не поднят."""
    try:
        with _socket.create_connection(('localhost', 8977), timeout=0.5) as s:
            s.sendall(_json.dumps(obj).encode('utf-8'))
            return _json.loads(s.recv(4096).decode('utf-8'))
    except OSError:
        return None


_probe = _ros_send({'ping': 1})
ros_link = W.Valid(value=_probe is not None,
                   readout='мост: ' + ('подключён' if _probe else 'не найден'))
ros_forward = W.Checkbox(False, description='отправлять панель в ROS-стенд')

if _probe is None:
    print('ROS-стенд не найден на localhost:8977 - это нормально, '
          'если он не запущен.')
    print('Запуск: docker compose up -d stand   (в каталоге docker/ '
          'репозитория ros-labs-stand)')
else:
    print('мост отвечает:', _probe)


def _forward_to_ros(_=None):
    if not ros_forward.value or not ros_link.value:
        return
    if mode.value.startswith('цель'):
        _ros_send({'goal': [tx.value, ty.value, tz.value],
                   'seed': [q1.value, q2.value, q3.value, q4.value]})
    else:
        r = _ros_send({'q': [q1.value, q2.value, q3.value, q4.value]})
        if r is not None and 'q' not in r and r.get('err'):
            print('мост:', r['err'])


for _wid in (q1, q2, q3, q4, tx, ty, tz):
    _wid.observe(_forward_to_ros, names='value')
mode.observe(lambda _=None: _forward_to_ros(), names='value')


def _forward_cam(_=None):
    if ros_forward.value and ros_link.value:
        _ros_send({'t_cam': t_cam.value})


# отдельная подписка: перелёт камеры всегда уходит одним числом
t_cam.observe(_forward_cam, names='value')
display(W.HBox([ros_forward, ros_link]))

ROS-стенд не найден на localhost:8977 - это нормально, если он не запущен.
Запуск: docker compose up -d stand   (в каталоге docker/ репозитория ros-labs-stand)


## Автотест конвейера (без панели, для воспроизводимости)

Прогоняет те же формулы в жёстких сценариях и сверяет с теорией из лабораторных.

In [10]:
# Автотест: жёсткие сверки с ЛР_01-06
# 1) якобиан 3D-руки: аналитический против численного
err_J = 0.0
for qq in rng.uniform(-2.0, 2.0, (40, 4)):
    J_an = arm_jac(qq)
    h = 1e-6
    J_num = np.zeros((3, 4))
    for j in range(4):
        qp, qm = qq.copy(), qq.copy()
        qp[j] += h; qm[j] -= h
        J_num[:, j] = (arm_fk(qp)[0][-1] - arm_fk(qm)[0][-1]) / (2 * h)
    err_J = max(err_J, np.abs(J_an - J_num).max())
print('якобиан: max || J_аналит - J_численный || на 40 конф. = %.2e' % err_J)
assert err_J < 1e-8

# 2) ЛР_02: кватернионы - round trip и геодезичность slerp
R = quat_to_R(R_to_quat(R0))
err_rt = np.abs(R - R0).max()
q_mid = slerp(Q_A, Q_B, 0.5)
ang_AB = np.degrees(2 * np.arccos(np.clip(abs(Q_A @ Q_B), -1, 1)))
ang_Am = np.degrees(2 * np.arccos(np.clip(abs(Q_A @ q_mid), -1, 1)))
err_geo = abs(ang_Am - ang_AB / 2)
err_ends = max(np.abs(quat_to_R(slerp(Q_A, Q_B, 0.0)) - R_A).max(),
               np.abs(quat_to_R(slerp(Q_A, Q_B, 1.0)) - R_B).max())
print('кват.:  round trip = %.2e, slerp концы = %.2e, середина %.1f/%.1f град (ош. %.1e)'
      % (err_rt, err_ends, ang_Am, ang_AB / 2, err_geo))
assert err_rt < 1e-12 and err_ends < 1e-12 and err_geo < 1e-9

# 3) ЛР_01: lookup через мир == прямое преобразование
P_cam = np.eye(4); P_cam[:3, :3], P_cam[:3, 3] = R0.T, -R0.T @ T0
T_wm = np.eye(4); T_wm[:3, 3] = MARKER_C
err_chain = np.abs((np.linalg.inv(P_cam) @ T_wm)[:3, 3]
                   - (R0 @ MARKER_C + T0)).max()
print('кадры:  lookup через мир == прямой: %.2e' % err_chain)
assert err_chain < 1e-12

# 4) ЛР_06 в 3D: IK сходится к случайным достижимым целям
fails = 0
for _ in range(20):
    q_star = rng.uniform(-2.0, 2.0, 4)
    target = arm_fk(q_star)[0][-1]
    _, hist, _ = ik3d_newton(target, q_star + rng.uniform(-0.4, 0.4, 4))
    if not hist[-1] < 1e-9:
        fails += 1
print('IK 3D:  20 случайных целей, не сошлось: %d' % fails)
assert fails <= 2

# 5) избыточность: ядро в 3D
q_sol, _, _ = ik3d_newton(np.array([0.45, 0.35, 0.55]),
                          [0.5, -0.5, 0.9, 0.6])
ns = np.linalg.svd(arm_jac(q_sol))[2][-1]
drift = [np.linalg.norm(arm_fk(q_sol + t * ns)[0][-1]
                        - arm_fk(q_sol)[0][-1]) for t in (0.05, 0.15)]
print('ядро:   сдвиг вдоль ядра -> смещение схвата %s м' % np.round(drift, 5))
assert drift[0] < 1e-3

# 6) граница рабочей зоны: cond(J) растёт
p_shoulder = np.array([0.0, 0.0, D_BASE])
v = np.array([1.0, 0.3, 0.15]); v /= np.linalg.norm(v)
cc = []
for d in (1e-1, 3e-3):
    qq, hh, c3 = ik3d_newton(p_shoulder + v * (REACH - d),
                              [0.0, 0.4, 0.6, 0.4])
    assert hh[-1] < 1e-8
    cc.append(c3[-1])
print('граница: cond(J) при d=0.1 -> %.0f, при d=0.003 -> %.0f' % (cc[0], cc[1]))
assert cc[1] > cc[0] * 2

# 7) ЛР_03: стерео-СКО против теории
uvL, zL = project(K, R0, T0, MARKER_C[None])
disp = F_PX * BASE_STEREO / zL[0]
noise = rng.normal(0, 0.25, 5000)
z_est = F_PX * BASE_STEREO / np.maximum(disp + noise, 1e-9)
rms = np.sqrt(np.mean((z_est - zL[0]) ** 2))
pred = zL[0] ** 2 / (F_PX * BASE_STEREO) * 0.25
print('стерео: СКО %.4f м, теория %.4f м' % (rms, pred))
assert 0.4 < rms / pred < 2.5

# 8) ЛР_04: RANSAC при 30% выбросов
obs_n = obs_clean + rng.normal(0, 0.2, obs_clean.shape)
inl_true = np.ones(len(obs_clean), bool)
bad = rng.choice(len(obs_clean), int(0.3 * len(obs_clean)), replace=False)
obs_n[bad] += rng.uniform(-150, 150, (len(bad), 2))
inl_true[bad] = False
Hh, inl = ransac_homography(M2, obs_n, seed=3)
dt30 = np.linalg.norm(pose_from_homography(Hh, K)[1]
                      - (R0 @ MARKER_C + T0))
print('AR: 30%% выбросов, dt = %.4f м, ложных inliers %d'
      % (dt30, int((inl & ~inl_true).sum())))
assert dt30 < 0.08

print()
print('АПОФЕОЗ ЛР_01-06: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

якобиан: max || J_аналит - J_численный || на 40 конф. = 2.30e-10
кват.:  round trip = 1.11e-16, slerp концы = 3.33e-16, середина 44.1/44.1 град (ош. 2.1e-14)
кадры:  lookup через мир == прямой: 4.44e-16
IK 3D:  20 случайных целей, не сошлось: 0
ядро:   сдвиг вдоль ядра -> смещение схвата [0.0002 0.002 ] м
граница: cond(J) при d=0.1 -> 5, при d=0.003 -> 30
стерео: СКО 0.0173 м, теория 0.0174 м
AR: 30% выбросов, dt = 0.0548 м, ложных inliers 0

АПОФЕОЗ ЛР_01-06: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
